In [3]:
# ============================================================
# ML4 - CELL 1
# Imports, Configuration & Logging
# ============================================================

import os
import json
import logging
import warnings

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------
os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(
            "logs/ml4_anomaly_detection.log",
            encoding="utf-8"
        ),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger("ML4")

logger.info("=" * 70)
logger.info("ML4 - Historical Behaviour Anomaly Detection")
logger.info("=" * 70)

# ------------------------------------------------------------
# MySQL Configuration
# ------------------------------------------------------------
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_DATABASE = "telecom_analytics"
MYSQL_USER = "root"
MYSQL_PASSWORD = "root"

MYSQL_URL = (
    f"mysql+pymysql://"
    f"{MYSQL_USER}:{MYSQL_PASSWORD}@"
    f"{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

# ------------------------------------------------------------
# ML4 Configuration - LOCKED
# ------------------------------------------------------------

# Activity column used for anomaly detection
ACTIVITY_COLUMN = "total_activity"

# Historical baseline
BASELINE_STATISTIC = "median"

# Baseline bucket
BASELINE_KEYS = ["grid_id", "hour_of_day"]

# Minimum historical observations required
MIN_HISTORY_COUNT = 3

# Anomaly threshold
ANOMALY_THRESHOLD = 0.50

# Direction labels
HIGH_LABEL = "HIGH"
LOW_LABEL = "LOW"
NORMAL_LABEL = "NORMAL"

# Operational action
ANOMALY_ACTION = "INVESTIGATE"

# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

ML4_DATASET_PATH = "data/ml4_anomaly_dataset.csv"
ML4_METADATA_PATH = "models/ml4_anomaly_metadata.json"

# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------
print("ML4 configuration loaded successfully.")
print("-" * 70)
print(f"Baseline statistic       : {BASELINE_STATISTIC}")
print(f"Baseline keys            : {BASELINE_KEYS}")
print(f"Minimum history count   : {MIN_HISTORY_COUNT}")
print(f"Anomaly threshold       : ±{ANOMALY_THRESHOLD:.0%}")
print(f"High anomaly            : deviation >= +{ANOMALY_THRESHOLD:.0%}")
print(f"Low anomaly             : deviation <= -{ANOMALY_THRESHOLD:.0%}")
print(f"Normal                   : within ±{ANOMALY_THRESHOLD:.0%}")
print(f"Operational action       : {ANOMALY_ACTION}")
print(f"Output dataset           : {ML4_DATASET_PATH}")
print(f"Metadata                 : {ML4_METADATA_PATH}")

logger.info("ML4 configuration initialized.")
logger.info(
    f"Baseline={BASELINE_STATISTIC}, "
    f"keys={BASELINE_KEYS}, "
    f"min_history={MIN_HISTORY_COUNT}, "
    f"threshold=±{ANOMALY_THRESHOLD:.0%}"
)

2026-09-05 16:24:43,057 | INFO | ======================================================================
2026-09-05 16:24:43,058 | INFO | ML4 - Historical Behaviour Anomaly Detection
2026-09-05 16:24:43,059 | INFO | ======================================================================
2026-09-05 16:24:43,061 | INFO | ML4 configuration initialized.
2026-09-05 16:24:43,062 | INFO | Baseline=median, keys=['grid_id', 'hour_of_day'], min_history=3, threshold=±50%


ML4 configuration loaded successfully.
----------------------------------------------------------------------
Baseline statistic       : median
Baseline keys            : ['grid_id', 'hour_of_day']
Minimum history count   : 3
Anomaly threshold       : ±50%
High anomaly            : deviation >= +50%
Low anomaly             : deviation <= -50%
Normal                   : within ±50%
Operational action       : INVESTIGATE
Output dataset           : data/ml4_anomaly_dataset.csv
Metadata                 : models/ml4_anomaly_metadata.json


In [4]:
# ============================================================
# ML4 - CELL 2
# Load Curated Hourly Network Activity from MySQL
# ============================================================

logger.info("=" * 70)
logger.info("ML4 CELL 2 - Loading curated network activity")
logger.info("=" * 70)

# ------------------------------------------------------------
# Create SQLAlchemy engine
# ------------------------------------------------------------
engine = create_engine(
    MYSQL_URL,
    pool_pre_ping=True
)

logger.info("MySQL SQLAlchemy engine created.")

# ------------------------------------------------------------
# Query curated activity
# ------------------------------------------------------------
activity_query = """
SELECT
    dg.grid_id,
    dt.timestamp AS timestamp,
    dt.hour_of_day,
    f.total_activity
FROM fact_network_activity f
JOIN dim_grid dg
    ON f.grid_key = dg.grid_key
JOIN dim_time dt
    ON f.time_key = dt.time_key
ORDER BY
    dt.timestamp,
    dg.grid_id
"""

logger.info("Executing activity query...")

ml4_activity_df = pd.read_sql(
    text(activity_query),
    engine
)

logger.info(
    f"Loaded {len(ml4_activity_df):,} activity rows."
)

# ------------------------------------------------------------
# Basic datatype preparation
# ------------------------------------------------------------
ml4_activity_df["timestamp"] = pd.to_datetime(
    ml4_activity_df["timestamp"]
)

ml4_activity_df["grid_id"] = ml4_activity_df["grid_id"].astype(int)

ml4_activity_df["hour_of_day"] = (
    ml4_activity_df["hour_of_day"].astype(int)
)

ml4_activity_df["total_activity"] = (
    pd.to_numeric(
        ml4_activity_df["total_activity"],
        errors="coerce"
    )
)

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------
print("\nML4 activity dataset loaded.")
print("-" * 70)

print(f"Rows              : {len(ml4_activity_df):,}")
print(f"Columns           : {len(ml4_activity_df.columns):,}")
print(
    f"Unique grids      : "
    f"{ml4_activity_df['grid_id'].nunique():,}"
)
print(
    f"Unique timestamps : "
    f"{ml4_activity_df['timestamp'].nunique():,}"
)

print(
    f"Timestamp range   : "
    f"{ml4_activity_df['timestamp'].min()} "
    f"→ "
    f"{ml4_activity_df['timestamp'].max()}"
)

print(
    f"Null total_activity : "
    f"{ml4_activity_df['total_activity'].isna().sum():,}"
)

print(
    f"Duplicate grid/timestamp rows : "
    f"{ml4_activity_df.duplicated(['grid_id', 'timestamp']).sum():,}"
)

print("\nHour-of-day distribution:")
print(
    ml4_activity_df["hour_of_day"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nActivity summary:")
print(
    ml4_activity_df["total_activity"]
    .describe()
    .to_string()
)

logger.info("ML4 activity validation completed.")

# ------------------------------------------------------------
# Preview
# ------------------------------------------------------------
print("\nFirst 10 rows:")
display(
    ml4_activity_df.head(10)
)

2026-09-05 16:24:47,626 | INFO | ======================================================================
2026-09-05 16:24:47,629 | INFO | ML4 CELL 2 - Loading curated network activity
2026-09-05 16:24:47,630 | INFO | ======================================================================
2026-09-05 16:24:47,632 | INFO | MySQL SQLAlchemy engine created.
2026-09-05 16:24:47,633 | INFO | Executing activity query...
2026-09-05 16:27:40,939 | INFO | Loaded 1,679,994 activity rows.



ML4 activity dataset loaded.
----------------------------------------------------------------------
Rows              : 1,679,994
Columns           : 4
Unique grids      : 10,000
Unique timestamps : 168
Timestamp range   : 2013-11-01 00:00:00 → 2013-11-07 23:00:00
Null total_activity : 0
Duplicate grid/timestamp rows : 0

Hour-of-day distribution:
hour_of_day
0     70000
1     70000
2     69999
3     69999
4     69997
5     69999
6     70000
7     70000
8     70000
9     70000
10    70000
11    70000
12    70000
13    70000
14    70000
15    70000
16    70000
17    70000
18    70000
19    70000
20    70000
21    70000
22    70000
23    70000

Activity summary:


2026-09-05 16:27:41,224 | INFO | ML4 activity validation completed.


count    1.679994e+06
mean     4.902707e+02
std      9.362976e+02
min      3.183000e-01
25%      8.246445e+01
50%      2.115289e+02
75%      5.017953e+02
max      3.798178e+04

First 10 rows:


,grid_id,timestamp,hour_of_day,total_activity
0,1,2013-11-01,0,62.0092
1,2,2013-11-01,0,62.1346
2,3,2013-11-01,0,62.2679
3,4,2013-11-01,0,61.6463
4,5,2013-11-01,0,55.9711
5,6,2013-11-01,0,62.2679
6,7,2013-11-01,0,62.2679
7,8,2013-11-01,0,62.2679
8,9,2013-11-01,0,62.2679
9,10,2013-11-01,0,36.0330


In [5]:
# ============================================================
# ML4 - CELL 3
# Temporal Structure & Historical Coverage Validation
# ============================================================

logger.info("=" * 70)
logger.info("ML4 CELL 3 - Validating temporal structure")
logger.info("=" * 70)

# ------------------------------------------------------------
# Sort explicitly by grid and timestamp
# ------------------------------------------------------------
ml4_activity_df = (
    ml4_activity_df
    .sort_values(
        ["grid_id", "timestamp"],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

logger.info("Activity data sorted by grid_id and timestamp.")

# ------------------------------------------------------------
# Derive date and hour independently from timestamp
# ------------------------------------------------------------
ml4_activity_df["date"] = (
    ml4_activity_df["timestamp"].dt.date
)

derived_hour = (
    ml4_activity_df["timestamp"].dt.hour
)

hour_mismatch = (
    ml4_activity_df["hour_of_day"] != derived_hour
).sum()

# ------------------------------------------------------------
# Check timestamp gaps within each grid
# ------------------------------------------------------------
ml4_activity_df["previous_timestamp"] = (
    ml4_activity_df
    .groupby("grid_id")["timestamp"]
    .shift(1)
)

ml4_activity_df["timestamp_gap"] = (
    ml4_activity_df["timestamp"]
    - ml4_activity_df["previous_timestamp"]
)

# Ignore first observation per grid
gap_mask = (
    ml4_activity_df["previous_timestamp"].notna()
    & (
        ml4_activity_df["timestamp_gap"]
        != pd.Timedelta(hours=1)
    )
)

gap_count = gap_mask.sum()

# ------------------------------------------------------------
# Check number of historical observations per grid/hour
# ------------------------------------------------------------
#
# This represents how many observations exist in each
# grid + hour-of-day bucket across the full dataset.
#
# Since the dataset spans 7 days, a complete bucket would
# normally have up to 7 observations.
# ------------------------------------------------------------
bucket_counts = (
    ml4_activity_df
    .groupby(["grid_id", "hour_of_day"])
    .size()
    .rename("bucket_count")
    .reset_index()
)

print("\nML4 temporal validation")
print("-" * 70)

print(f"Hour-of-day mismatches       : {hour_mismatch:,}")
print(f"Unexpected timestamp gaps    : {gap_count:,}")

print(
    f"Grid/hour buckets            : "
    f"{len(bucket_counts):,}"
)

print(
    f"Minimum bucket observations  : "
    f"{bucket_counts['bucket_count'].min()}"
)

print(
    f"Maximum bucket observations  : "
    f"{bucket_counts['bucket_count'].max()}"
)

print(
    f"Mean bucket observations     : "
    f"{bucket_counts['bucket_count'].mean():.4f}"
)

print("\nHistorical observation count distribution:")
print(
    bucket_counts["bucket_count"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# Verify the expected number of grid/hour buckets
# ------------------------------------------------------------
expected_buckets = 10_000 * 24

print("\nBucket completeness")
print("-" * 70)
print(f"Expected grid/hour buckets   : {expected_buckets:,}")
print(f"Actual grid/hour buckets     : {len(bucket_counts):,}")

missing_buckets = (
    expected_buckets - len(bucket_counts)
)

print(f"Missing grid/hour buckets    : {missing_buckets:,}")

# ------------------------------------------------------------
# Show buckets with fewer than 3 observations
# ------------------------------------------------------------
low_history_buckets = (
    bucket_counts[
        bucket_counts["bucket_count"] < MIN_HISTORY_COUNT
    ]
    .sort_values(
        ["bucket_count", "grid_id", "hour_of_day"]
    )
)

print("\nBuckets with insufficient historical observations")
print("-" * 70)
print(
    f"Buckets with < {MIN_HISTORY_COUNT} observations : "
    f"{len(low_history_buckets):,}"
)

if len(low_history_buckets) > 0:
    display(
        low_history_buckets.head(20)
    )

# ------------------------------------------------------------
# Cleanup temporary validation columns
# ------------------------------------------------------------
ml4_activity_df.drop(
    columns=[
        "previous_timestamp",
        "timestamp_gap"
    ],
    inplace=True
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------
assert hour_mismatch == 0, (
    "hour_of_day does not consistently match timestamp."
)

logger.info(
    f"Hour-of-day validation passed: {hour_mismatch} mismatches."
)

logger.info(
    f"Found {len(bucket_counts):,} grid/hour buckets."
)

logger.info(
    f"Bucket observation range: "
    f"{bucket_counts['bucket_count'].min()}-"
    f"{bucket_counts['bucket_count'].max()}"
)

logger.info(
    f"Buckets with fewer than {MIN_HISTORY_COUNT} "
    f"historical observations: "
    f"{len(low_history_buckets):,}"
)

if gap_count == 0:
    logger.info("No unexpected hourly gaps detected within grids.")
else:
    logger.warning(
        f"Detected {gap_count:,} unexpected timestamp gaps."
    )

logger.info("ML4 temporal validation completed successfully.")

2026-09-05 16:29:07,069 | INFO | ======================================================================
2026-09-05 16:29:07,070 | INFO | ML4 CELL 3 - Validating temporal structure
2026-09-05 16:29:07,072 | INFO | ======================================================================
2026-09-05 16:29:07,204 | INFO | Activity data sorted by grid_id and timestamp.
2026-09-05 16:29:07,723 | INFO | Hour-of-day validation passed: 0 mismatches.
2026-09-05 16:29:07,726 | INFO | Found 240,000 grid/hour buckets.
2026-09-05 16:29:07,729 | INFO | Bucket observation range: 4-7
2026-09-05 16:29:07,730 | INFO | Buckets with fewer than 3 historical observations: 0
2026-09-05 16:29:07,731 | WARNING | Detected 5 unexpected timestamp gaps.
2026-09-05 16:29:07,732 | INFO | ML4 temporal validation completed successfully.



ML4 temporal validation
----------------------------------------------------------------------
Hour-of-day mismatches       : 0
Unexpected timestamp gaps    : 5
Grid/hour buckets            : 240,000
Minimum bucket observations  : 4
Maximum bucket observations  : 7
Mean bucket observations     : 7.0000

Historical observation count distribution:
bucket_count
4         1
6         3
7    239996

Bucket completeness
----------------------------------------------------------------------
Expected grid/hour buckets   : 240,000
Actual grid/hour buckets     : 240,000
Missing grid/hour buckets    : 0

Buckets with insufficient historical observations
----------------------------------------------------------------------
Buckets with < 3 observations : 0


In [12]:
# ============================================================
# ML4 - CELL 4 FINAL
# Historical Baseline - Explicit Historical Lookup
# ============================================================

logger.info("=" * 70)
logger.info("ML4 CELL 4 - Final historical baseline calculation")
logger.info("=" * 70)

# ------------------------------------------------------------
# Work from a clean copy
# ------------------------------------------------------------
ml4_activity_df = (
    ml4_activity_df
    .drop(
        columns=[
            "historical_baseline",
            "baseline_count"
        ],
        errors="ignore"
    )
    .copy()
)

# ------------------------------------------------------------
# Sort chronologically
# ------------------------------------------------------------
ml4_activity_df = (
    ml4_activity_df
    .sort_values(
        ["grid_id", "hour_of_day", "timestamp"],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

logger.info(
    "Activity data sorted by grid_id, hour_of_day and timestamp."
)

# ------------------------------------------------------------
# Create historical position within each grid/hour bucket
#
# position:
#   0 = first occurrence
#   1 = second occurrence
#   2 = third occurrence
#   ...
#
# Therefore:
#   historical_count = position
#
# because the current observation is excluded.
# ------------------------------------------------------------
ml4_activity_df["baseline_count"] = (
    ml4_activity_df
    .groupby(
        ["grid_id", "hour_of_day"],
        sort=False
    )
    .cumcount()
)

# ------------------------------------------------------------
# Calculate historical median
#
# IMPORTANT:
#
# We first shift the activity value by one position.
#
# Therefore, for:
#
#   Nov 7 18:00
#
# the shifted historical values contain:
#
#   Nov 1 18:00
#   Nov 2 18:00
#   ...
#   Nov 6 18:00
#
# but NOT Nov 7 18:00.
# ------------------------------------------------------------
ml4_activity_df["_previous_activity"] = (
    ml4_activity_df
    .groupby(
        ["grid_id", "hour_of_day"],
        sort=False
    )[ACTIVITY_COLUMN]
    .shift(1)
)

# ------------------------------------------------------------
# Historical median
# ------------------------------------------------------------
ml4_activity_df["historical_baseline"] = (
    ml4_activity_df
    .groupby(
        ["grid_id", "hour_of_day"],
        sort=False
    )["_previous_activity"]
    .transform(
        lambda x: x.expanding(
            min_periods=1
        ).median()
    )
)

# ------------------------------------------------------------
# Enforce minimum history
# ------------------------------------------------------------
ml4_activity_df.loc[
    ml4_activity_df["baseline_count"] < MIN_HISTORY_COUNT,
    "historical_baseline"
] = np.nan

# ------------------------------------------------------------
# Remove temporary column
# ------------------------------------------------------------
ml4_activity_df.drop(
    columns=["_previous_activity"],
    inplace=True
)

# ------------------------------------------------------------
# Validation summary
# ------------------------------------------------------------
print("\nML4 historical baseline calculated.")
print("-" * 70)

print(
    f"Rows              : "
    f"{len(ml4_activity_df):,}"
)

print(
    f"Valid baselines   : "
    f"{ml4_activity_df['historical_baseline'].notna().sum():,}"
)

print(
    f"Missing baselines : "
    f"{ml4_activity_df['historical_baseline'].isna().sum():,}"
)

print(
    f"Baseline count min: "
    f"{ml4_activity_df['baseline_count'].min()}"
)

print(
    f"Baseline count max: "
    f"{ml4_activity_df['baseline_count'].max()}"
)

print("\nComplete baseline-count distribution:")
print(
    ml4_activity_df["baseline_count"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# Expected count distribution
# ------------------------------------------------------------
print("\nExpected interpretation:")
print("  0 = first observation in grid/hour bucket")
print("  1 = one historical observation")
print("  2 = two historical observations")
print("  3 = three historical observations -> first valid baseline")
print("  4 = four historical observations")
print("  5 = five historical observations")
print("  6 = six historical observations")

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

# Every bucket must start at historical count 0.
first_counts = (
    ml4_activity_df
    .groupby(
        ["grid_id", "hour_of_day"]
    )["baseline_count"]
    .min()
)

assert (
    (first_counts == 0).all()
), "Every grid/hour bucket should start with count 0."

# Every valid baseline must have at least 3 historical values.
assert (
    ml4_activity_df.loc[
        ml4_activity_df["historical_baseline"].notna(),
        "baseline_count"
    ] >= MIN_HISTORY_COUNT
).all(), (
    "Found a valid baseline with insufficient history."
)

# Every invalid baseline must have fewer than 3 observations.
assert (
    ml4_activity_df.loc[
        ml4_activity_df["historical_baseline"].isna(),
        "baseline_count"
    ] < MIN_HISTORY_COUNT
).all(), (
    "Found a missing baseline despite sufficient history."
)

logger.info(
    "Historical count structure validated."
)

logger.info(
    "Minimum-history requirement validated."
)

logger.info(
    "ML4 CELL 4 FINAL completed successfully."
)

2026-09-05 16:46:34,733 | INFO | ======================================================================
2026-09-05 16:46:34,734 | INFO | ML4 CELL 4 - Final historical baseline calculation
2026-09-05 16:46:34,735 | INFO | ======================================================================
2026-09-05 16:46:34,876 | INFO | Activity data sorted by grid_id, hour_of_day and timestamp.
2026-09-05 16:47:58,323 | INFO | Historical count structure validated.
2026-09-05 16:47:58,325 | INFO | Minimum-history requirement validated.
2026-09-05 16:47:58,326 | INFO | ML4 CELL 4 FINAL completed successfully.



ML4 historical baseline calculated.
----------------------------------------------------------------------
Rows              : 1,679,994
Valid baselines   : 959,994
Missing baselines : 720,000
Baseline count min: 0
Baseline count max: 6

Complete baseline-count distribution:
baseline_count
0    240000
1    240000
2    240000
3    240000
4    239999
5    239999
6    239996

Expected interpretation:
  0 = first observation in grid/hour bucket
  1 = one historical observation
  2 = two historical observations
  3 = three historical observations -> first valid baseline
  4 = four historical observations
  5 = five historical observations
  6 = six historical observations


In [13]:
# ============================================================
# ML4 - CELL 5 FINAL
# Historical Baseline Integrity Validation
# ============================================================

logger.info("=" * 70)
logger.info("ML4 CELL 5 - Final baseline integrity validation")
logger.info("=" * 70)

# ------------------------------------------------------------
# Create a clean validation dataframe
# ------------------------------------------------------------
validation_df = (
    ml4_activity_df[
        [
            "grid_id",
            "hour_of_day",
            "timestamp",
            "total_activity",
            "historical_baseline",
            "baseline_count"
        ]
    ]
    .sort_values(
        ["grid_id", "hour_of_day", "timestamp"],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Expected historical count
#
# For each grid + hour bucket:
#
# observation 1 -> 0 previous observations
# observation 2 -> 1
# observation 3 -> 2
# observation 4 -> 3
# ...
# observation 7 -> 6
# ------------------------------------------------------------
validation_df["expected_history_count"] = (
    validation_df
    .groupby(
        ["grid_id", "hour_of_day"],
        sort=False
    )
    .cumcount()
)

# ------------------------------------------------------------
# IMPORTANT:
#
# Cell 4's baseline_count represents the number of historical
# values available after the shift.
#
# We therefore compare it against the expected count ONLY
# for rows where a baseline is valid.
# ------------------------------------------------------------
valid_mask = (
    validation_df["historical_baseline"].notna()
)

valid_validation = validation_df[valid_mask].copy()

# ------------------------------------------------------------
# Count validation
# ------------------------------------------------------------
count_mismatch_mask = (
    valid_validation["baseline_count"]
    != valid_validation["expected_history_count"]
)

count_mismatches = (
    count_mismatch_mask.sum()
)

# ------------------------------------------------------------
# Baseline value validation
#
# Independently calculate the median of all observations
# strictly before the current timestamp within the same
# grid/hour bucket.
#
# We do this on a manageable random sample.
# ------------------------------------------------------------
sample_size = min(1000, len(valid_validation))

validation_sample = (
    valid_validation
    .sample(
        n=sample_size,
        random_state=42
    )
    .copy()
)

independent_medians = []

for row in validation_sample.itertuples(index=False):

    previous_values = (
        ml4_activity_df[
            (ml4_activity_df["grid_id"] == row.grid_id)
            & (
                ml4_activity_df["hour_of_day"]
                == row.hour_of_day
            )
            & (
                ml4_activity_df["timestamp"]
                < row.timestamp
            )
        ][ACTIVITY_COLUMN]
    )

    independent_medians.append(
        previous_values.median()
    )

validation_sample["independent_median"] = (
    independent_medians
)

validation_sample["baseline_difference"] = (
    validation_sample["historical_baseline"]
    - validation_sample["independent_median"]
).abs()

max_baseline_difference = (
    validation_sample["baseline_difference"].max()
)

nonzero_baseline_differences = (
    (
        validation_sample["baseline_difference"]
        > 1e-10
    ).sum()
)

# ------------------------------------------------------------
# Verify minimum-history rule
# ------------------------------------------------------------
invalid_history_rows = (
    valid_validation["baseline_count"]
    < MIN_HISTORY_COUNT
).sum()

# ------------------------------------------------------------
# Verify no baseline exists before minimum history
# ------------------------------------------------------------
invalid_early_baselines = (
    validation_df[
        validation_df["historical_baseline"].notna()
        & (
            validation_df["expected_history_count"]
            < MIN_HISTORY_COUNT
        )
    ]
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------
print("\nML4 baseline integrity validation")
print("-" * 70)

print(
    f"Total activity rows             : "
    f"{len(validation_df):,}"
)

print(
    f"Rows with valid baseline        : "
    f"{valid_mask.sum():,}"
)

print(
    f"Rows without valid baseline     : "
    f"{(~valid_mask).sum():,}"
)

print(
    f"Expected history count range   : "
    f"{validation_df['expected_history_count'].min()} "
    f"→ "
    f"{validation_df['expected_history_count'].max()}"
)

print(
    f"Valid baseline count range     : "
    f"{valid_validation['baseline_count'].min()} "
    f"→ "
    f"{valid_validation['baseline_count'].max()}"
)

print(
    f"Baseline count mismatches       : "
    f"{count_mismatches:,}"
)

print(
    f"Rows violating minimum history : "
    f"{invalid_history_rows:,}"
)

print(
    f"Early baselines (< minimum)     : "
    f"{len(invalid_early_baselines):,}"
)

print(
    f"Independent sample size         : "
    f"{sample_size:,}"
)

print(
    f"Maximum baseline difference     : "
    f"{max_baseline_difference:.12f}"
)

print(
    f"Non-zero baseline differences   : "
    f"{nonzero_baseline_differences:,}"
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------
assert count_mismatches == 0, (
    "Baseline count does not match chronological history."
)

assert invalid_history_rows == 0, (
    "Some valid baselines have insufficient history."
)

assert len(invalid_early_baselines) == 0, (
    "Baseline exists before minimum history requirement."
)

assert nonzero_baseline_differences == 0, (
    "Calculated baseline differs from independent median."
)

logger.info(
    "Baseline count validation PASSED."
)

logger.info(
    "Minimum-history validation PASSED."
)

logger.info(
    "Independent median validation PASSED."
)

logger.info(
    "ML4 baseline integrity is confirmed."
)

# ------------------------------------------------------------
# Show examples
# ------------------------------------------------------------
print("\nSample validated baseline rows:")
display(
    validation_sample[
        [
            "grid_id",
            "hour_of_day",
            "timestamp",
            "total_activity",
            "historical_baseline",
            "independent_median",
            "baseline_count",
            "expected_history_count"
        ]
    ].head(10)
)

# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------
del validation_df
del valid_validation
del validation_sample

logger.info(
    "ML4 CELL 5 FINAL completed successfully."
)

2026-09-05 16:48:57,016 | INFO | ======================================================================
2026-09-05 16:48:57,018 | INFO | ML4 CELL 5 - Final baseline integrity validation
2026-09-05 16:48:57,021 | INFO | ======================================================================
2026-09-05 16:49:15,550 | INFO | Baseline count validation PASSED.
2026-09-05 16:49:15,553 | INFO | Minimum-history validation PASSED.
2026-09-05 16:49:15,555 | INFO | Independent median validation PASSED.
2026-09-05 16:49:15,558 | INFO | ML4 baseline integrity is confirmed.



ML4 baseline integrity validation
----------------------------------------------------------------------
Total activity rows             : 1,679,994
Rows with valid baseline        : 959,994
Rows without valid baseline     : 720,000
Expected history count range   : 0 → 6
Valid baseline count range     : 3 → 6
Baseline count mismatches       : 0
Rows violating minimum history : 0
Early baselines (< minimum)     : 0
Independent sample size         : 1,000
Maximum baseline difference     : 0.000000000000
Non-zero baseline differences   : 0

Sample validated baseline rows:


,grid_id,hour_of_day,timestamp,total_activity,historical_baseline,independent_median,baseline_count,expected_history_count
729424,4342,19,2013-11-04 19:00:00,1736.8915,1456.91970,1456.91970,3,3
403861,2404,22,2013-11-04 22:00:00,107.9096,98.27140,98.27140,3,3
951985,5667,14,2013-11-06 14:00:00,5326.2874,3230.24560,3230.24560,5,5
860003,5120,1,2013-11-05 01:00:00,98.1471,148.09545,148.09545,4,4
946736,5636,8,2013-11-07 08:00:00,716.9854,548.17375,548.17375,6,6
533063,3173,23,2013-11-07 23:00:00,118.7490,112.39915,112.39915,6,6
1405914,8369,13,2013-11-06 13:00:00,1402.0134,811.57320,811.57320,5,5
1344273,8002,15,2013-11-07 15:00:00,273.1372,259.78545,259.78545,6,6
740375,4407,23,2013-11-07 23:00:00,64.9616,71.40300,71.40300,6,6
838380,4991,8,2013-11-05 08:00:00,3083.3699,1495.87750,1495.87750,4,4


2026-09-05 16:49:15,595 | INFO | ML4 CELL 5 FINAL completed successfully.


In [14]:
# ============================================================
# ML4 - CELL 6
# Deviation & Anomaly Score Calculation
# ============================================================

logger.info("=" * 70)
logger.info("ML4 CELL 6 - Calculating deviation and anomaly score")
logger.info("=" * 70)

# ------------------------------------------------------------
# Ensure numeric types
# ------------------------------------------------------------
ml4_activity_df["total_activity"] = pd.to_numeric(
    ml4_activity_df["total_activity"],
    errors="coerce"
)

ml4_activity_df["historical_baseline"] = pd.to_numeric(
    ml4_activity_df["historical_baseline"],
    errors="coerce"
)

ml4_activity_df["baseline_count"] = (
    ml4_activity_df["baseline_count"].astype(int)
)

# ------------------------------------------------------------
# Identify valid baseline rows
# ------------------------------------------------------------
valid_baseline_mask = (
    ml4_activity_df["historical_baseline"].notna()
    & (
        ml4_activity_df["baseline_count"]
        >= MIN_HISTORY_COUNT
    )
)

# ------------------------------------------------------------
# Identify zero baselines
# ------------------------------------------------------------
zero_baseline_mask = (
    valid_baseline_mask
    & (
        ml4_activity_df["historical_baseline"] == 0
    )
)

zero_baseline_count = zero_baseline_mask.sum()

logger.info(
    f"Valid baseline rows: "
    f"{valid_baseline_mask.sum():,}"
)

logger.info(
    f"Valid rows with zero baseline: "
    f"{zero_baseline_count:,}"
)

# ------------------------------------------------------------
# Initialize deviation
# ------------------------------------------------------------
ml4_activity_df["deviation"] = np.nan

# ------------------------------------------------------------
# Calculate signed percentage deviation
#
# Formula:
#
# (current - baseline) / baseline
#
# Only calculate where:
#   baseline exists
#   baseline_count >= minimum
#   baseline != 0
# ------------------------------------------------------------
calculation_mask = (
    valid_baseline_mask
    & (~zero_baseline_mask)
)

ml4_activity_df.loc[
    calculation_mask,
    "deviation"
] = (
    (
        ml4_activity_df.loc[
            calculation_mask,
            "total_activity"
        ]
        -
        ml4_activity_df.loc[
            calculation_mask,
            "historical_baseline"
        ]
    )
    /
    ml4_activity_df.loc[
        calculation_mask,
        "historical_baseline"
    ]
)

# ------------------------------------------------------------
# Calculate absolute anomaly score
# ------------------------------------------------------------
ml4_activity_df["anomaly_score"] = (
    ml4_activity_df["deviation"].abs()
)

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

# Recalculate independently
independent_deviation = (
    (
        ml4_activity_df.loc[
            calculation_mask,
            "total_activity"
        ]
        -
        ml4_activity_df.loc[
            calculation_mask,
            "historical_baseline"
        ]
    )
    /
    ml4_activity_df.loc[
        calculation_mask,
        "historical_baseline"
    ]
)

deviation_difference = (
    ml4_activity_df.loc[
        calculation_mask,
        "deviation"
    ]
    -
    independent_deviation
).abs()

max_deviation_difference = (
    deviation_difference.max()
    if len(deviation_difference) > 0
    else 0.0
)

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------
print("\nML4 deviation calculation")
print("-" * 70)

print(
    f"Total rows                  : "
    f"{len(ml4_activity_df):,}"
)

print(
    f"Valid baseline rows         : "
    f"{valid_baseline_mask.sum():,}"
)

print(
    f"Rows used for calculation   : "
    f"{calculation_mask.sum():,}"
)

print(
    f"Zero-baseline rows          : "
    f"{zero_baseline_count:,}"
)

print(
    f"Rows with deviation         : "
    f"{ml4_activity_df['deviation'].notna().sum():,}"
)

print(
    f"Rows without deviation      : "
    f"{ml4_activity_df['deviation'].isna().sum():,}"
)

print(
    f"Maximum calculation error   : "
    f"{max_deviation_difference:.12f}"
)

# ------------------------------------------------------------
# Deviation statistics
# ------------------------------------------------------------
print("\nSigned deviation statistics:")
print(
    ml4_activity_df["deviation"]
    .describe()
    .to_string()
)

print("\nAbsolute anomaly-score statistics:")
print(
    ml4_activity_df["anomaly_score"]
    .describe()
    .to_string()
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------
assert max_deviation_difference < 1e-10, (
    "Deviation calculation failed independent validation."
)

assert (
    ml4_activity_df.loc[
        ~calculation_mask,
        "deviation"
    ].isna().all()
), (
    "Deviation exists for a row without a valid baseline."
)

assert (
    ml4_activity_df.loc[
        calculation_mask,
        "anomaly_score"
    ]
    >= 0
).all(), (
    "Anomaly score contains negative values."
)

logger.info(
    "Deviation calculation independently validated."
)

logger.info(
    "Anomaly score calculation validated."
)

logger.info(
    "ML4 CELL 6 completed successfully."
)

# ------------------------------------------------------------
# Preview
# ------------------------------------------------------------
print("\nSample ML4 deviation calculations:")
display(
    ml4_activity_df[
        [
            "grid_id",
            "timestamp",
            "hour_of_day",
            "total_activity",
            "historical_baseline",
            "baseline_count",
            "deviation",
            "anomaly_score"
        ]
    ]
    .dropna(subset=["deviation"])
    .head(15)
)

2026-09-05 16:53:36,453 | INFO | ======================================================================
2026-09-05 16:53:36,455 | INFO | ML4 CELL 6 - Calculating deviation and anomaly score
2026-09-05 16:53:36,456 | INFO | ======================================================================
2026-09-05 16:53:36,482 | INFO | Valid baseline rows: 959,994
2026-09-05 16:53:36,484 | INFO | Valid rows with zero baseline: 0
2026-09-05 16:53:36,768 | INFO | Deviation calculation independently validated.



ML4 deviation calculation
----------------------------------------------------------------------
Total rows                  : 1,679,994
Valid baseline rows         : 959,994
Rows used for calculation   : 959,994
Zero-baseline rows          : 0
Rows with deviation         : 959,994
Rows without deviation      : 720,000
Maximum calculation error   : 0.000000000000

Signed deviation statistics:
count    959994.000000
mean          0.279660
std           0.612364
min          -0.976112
25%          -0.007120
50%           0.135746
75%           0.369376
max          24.867596

Absolute anomaly-score statistics:
count    959994.000000
mean          0.349028
std           0.575656
min           0.000000
25%           0.075310
50%           0.179283
75%           0.391709
max          24.867596


2026-09-05 16:53:36,769 | INFO | Anomaly score calculation validated.
2026-09-05 16:53:36,769 | INFO | ML4 CELL 6 completed successfully.



Sample ML4 deviation calculations:


,grid_id,timestamp,hour_of_day,total_activity,historical_baseline,baseline_count,deviation,anomaly_score
3,1,2013-11-04 00:00:00,0,52.5542,57.48460,3,-0.085769,0.085769
4,1,2013-11-05 00:00:00,0,59.9878,55.01940,4,0.090303,0.090303
5,1,2013-11-06 00:00:00,0,54.1378,57.48460,5,-0.058221,0.058221
6,1,2013-11-07 00:00:00,0,59.6934,55.81120,6,0.069560,0.069560
10,1,2013-11-04 01:00:00,1,49.8515,46.36540,3,0.075188,0.075188
11,1,2013-11-05 01:00:00,1,51.3259,47.30515,4,0.084996,0.084996
12,1,2013-11-06 01:00:00,1,44.0278,48.24490,5,-0.087410,0.087410
13,1,2013-11-07 01:00:00,1,44.6076,47.30515,6,-0.057024,0.057024
17,1,2013-11-04 02:00:00,2,36.2455,41.65060,3,-0.129772,0.129772
18,1,2013-11-05 02:00:00,2,47.1223,41.25400,4,0.142248,0.142248


In [15]:
# ============================================================
# ML4 - CELL 7
# Anomaly Direction & Anomaly Flag
# ============================================================

logger.info("=" * 70)
logger.info("ML4 CELL 7 - Classifying anomaly direction")
logger.info("=" * 70)

# ------------------------------------------------------------
# Initialize direction
# ------------------------------------------------------------
ml4_activity_df["anomaly_direction"] = "UNKNOWN"

# ------------------------------------------------------------
# Valid deviation mask
# ------------------------------------------------------------
valid_deviation_mask = (
    ml4_activity_df["deviation"].notna()
)

# ------------------------------------------------------------
# HIGH anomaly
# ------------------------------------------------------------
high_anomaly_mask = (
    valid_deviation_mask
    & (
        ml4_activity_df["deviation"]
        >= ANOMALY_THRESHOLD
    )
)

# ------------------------------------------------------------
# LOW anomaly
# ------------------------------------------------------------
low_anomaly_mask = (
    valid_deviation_mask
    & (
        ml4_activity_df["deviation"]
        <= -ANOMALY_THRESHOLD
    )
)

# ------------------------------------------------------------
# NORMAL behaviour
# ------------------------------------------------------------
normal_mask = (
    valid_deviation_mask
    & (~high_anomaly_mask)
    & (~low_anomaly_mask)
)

# ------------------------------------------------------------
# Assign direction
# ------------------------------------------------------------
ml4_activity_df.loc[
    high_anomaly_mask,
    "anomaly_direction"
] = HIGH_LABEL

ml4_activity_df.loc[
    low_anomaly_mask,
    "anomaly_direction"
] = LOW_LABEL

ml4_activity_df.loc[
    normal_mask,
    "anomaly_direction"
] = NORMAL_LABEL

# ------------------------------------------------------------
# Create anomaly flag
#
# 1 = HIGH or LOW anomaly
# 0 = NORMAL or UNKNOWN
# ------------------------------------------------------------
ml4_activity_df["anomaly_flag"] = (
    high_anomaly_mask | low_anomaly_mask
).astype(int)

# ------------------------------------------------------------
# Validate classification
# ------------------------------------------------------------

# HIGH rows must meet threshold
assert (
    ml4_activity_df.loc[
        high_anomaly_mask,
        "deviation"
    ] >= ANOMALY_THRESHOLD
).all()

# LOW rows must meet threshold
assert (
    ml4_activity_df.loc[
        low_anomaly_mask,
        "deviation"
    ] <= -ANOMALY_THRESHOLD
).all()

# NORMAL rows must be inside threshold
assert (
    ml4_activity_df.loc[
        normal_mask,
        "deviation"
    ].abs() < ANOMALY_THRESHOLD
).all()

# UNKNOWN rows must have no deviation
assert (
    ml4_activity_df.loc[
        ml4_activity_df["anomaly_direction"] == "UNKNOWN",
        "deviation"
    ].isna()
).all()

# Anomaly flag must exactly match HIGH/LOW
expected_flag = (
    ml4_activity_df["anomaly_direction"]
    .isin([HIGH_LABEL, LOW_LABEL])
    .astype(int)
)

assert (
    ml4_activity_df["anomaly_flag"]
    == expected_flag
).all()

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

direction_counts = (
    ml4_activity_df["anomaly_direction"]
    .value_counts()
    .reindex(
        [
            HIGH_LABEL,
            LOW_LABEL,
            NORMAL_LABEL,
            "UNKNOWN"
        ],
        fill_value=0
    )
)

flag_counts = (
    ml4_activity_df["anomaly_flag"]
    .value_counts()
    .sort_index()
)

print("\nML4 anomaly classification")
print("-" * 70)

print("Direction distribution:")
print(
    direction_counts.to_string()
)

print("\nAnomaly flag distribution:")
print(
    flag_counts.to_string()
)

print("\nAnomaly percentages:")
for direction, count in direction_counts.items():
    percentage = (
        count / len(ml4_activity_df) * 100
    )

    print(
        f"  {direction:8s}: "
        f"{count:>10,} "
        f"({percentage:6.2f}%)"
    )

# ------------------------------------------------------------
# High / Low anomaly statistics
# ------------------------------------------------------------

print("\nHIGH anomaly deviation statistics:")

if high_anomaly_mask.sum() > 0:
    print(
        ml4_activity_df.loc[
            high_anomaly_mask,
            "deviation"
        ].describe().to_string()
    )
else:
    print("No HIGH anomalies detected.")

print("\nLOW anomaly deviation statistics:")

if low_anomaly_mask.sum() > 0:
    print(
        ml4_activity_df.loc[
            low_anomaly_mask,
            "deviation"
        ].describe().to_string()
    )
else:
    print("No LOW anomalies detected.")

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------

logger.info(
    f"HIGH anomalies: "
    f"{high_anomaly_mask.sum():,}"
)

logger.info(
    f"LOW anomalies: "
    f"{low_anomaly_mask.sum():,}"
)

logger.info(
    f"NORMAL observations: "
    f"{normal_mask.sum():,}"
)

logger.info(
    f"UNKNOWN observations: "
    f"{(~valid_deviation_mask).sum():,}"
)

logger.info(
    "ML4 anomaly direction and flag validation passed."
)

logger.info(
    "ML4 CELL 7 completed successfully."
)

# ------------------------------------------------------------
# Preview anomalies
# ------------------------------------------------------------

print("\nSample HIGH anomalies:")
display(
    ml4_activity_df.loc[
        high_anomaly_mask,
        [
            "grid_id",
            "timestamp",
            "hour_of_day",
            "total_activity",
            "historical_baseline",
            "baseline_count",
            "deviation",
            "anomaly_score",
            "anomaly_direction",
            "anomaly_flag"
        ]
    ]
    .sort_values(
        "anomaly_score",
        ascending=False
    )
    .head(10)
)

print("\nSample LOW anomalies:")
display(
    ml4_activity_df.loc[
        low_anomaly_mask,
        [
            "grid_id",
            "timestamp",
            "hour_of_day",
            "total_activity",
            "historical_baseline",
            "baseline_count",
            "deviation",
            "anomaly_score",
            "anomaly_direction",
            "anomaly_flag"
        ]
    ]
    .sort_values(
        "anomaly_score",
        ascending=False
    )
    .head(10)
)

2026-09-05 16:55:28,147 | INFO | ======================================================================
2026-09-05 16:55:28,149 | INFO | ML4 CELL 7 - Classifying anomaly direction
2026-09-05 16:55:28,149 | INFO | ======================================================================
2026-09-05 16:55:28,410 | INFO | HIGH anomalies: 174,321
2026-09-05 16:55:28,412 | INFO | LOW anomalies: 7,888
2026-09-05 16:55:28,414 | INFO | NORMAL observations: 777,785
2026-09-05 16:55:28,416 | INFO | UNKNOWN observations: 720,000
2026-09-05 16:55:28,420 | INFO | ML4 anomaly direction and flag validation passed.
2026-09-05 16:55:28,421 | INFO | ML4 CELL 7 completed successfully.



ML4 anomaly classification
----------------------------------------------------------------------
Direction distribution:
anomaly_direction
HIGH       174321
LOW          7888
NORMAL     777785
UNKNOWN    720000

Anomaly flag distribution:
anomaly_flag
0    1497785
1     182209

Anomaly percentages:
  HIGH    :    174,321 ( 10.38%)
  LOW     :      7,888 (  0.47%)
  NORMAL  :    777,785 ( 46.30%)
  UNKNOWN :    720,000 ( 42.86%)

HIGH anomaly deviation statistics:
count    174321.000000
mean          1.157380
std           0.970981
min           0.500000
25%           0.636334
50%           0.850629
75%           1.298130
max          24.867596

LOW anomaly deviation statistics:
count    7888.000000
mean       -0.659885
std         0.114709
min        -0.976112
25%        -0.741679
50%        -0.640932
75%        -0.560924
max        -0.500000

Sample HIGH anomalies:


,grid_id,timestamp,hour_of_day,total_activity,historical_baseline,baseline_count,deviation,anomaly_score,anomaly_direction,anomaly_flag
768316,4574,2013-11-04 07:00:00,7,311.8856,12.05700,3,24.867596,24.867596,HIGH,1
768309,4574,2013-11-04 06:00:00,6,291.6454,11.58220,3,24.180484,24.180484,HIGH,1
1184643,7052,2013-11-05 11:00:00,11,9631.5100,411.12445,4,22.427237,22.427237,HIGH,1
1297561,7724,2013-11-06 14:00:00,14,15322.9761,662.46070,5,22.130393,22.130393,HIGH,1
1184656,7052,2013-11-04 13:00:00,13,7938.9004,353.19810,3,21.477189,21.477189,HIGH,1
1184635,7052,2013-11-04 10:00:00,10,7752.0294,359.27060,3,20.577133,20.577133,HIGH,1
1184663,7052,2013-11-04 14:00:00,14,7923.4171,370.44980,3,20.388639,20.388639,HIGH,1
1297554,7724,2013-11-06 13:00:00,13,14731.7364,694.76140,5,20.204023,20.204023,HIGH,1
768302,4574,2013-11-04 05:00:00,5,277.8204,13.10530,3,20.199087,20.199087,HIGH,1
1184649,7052,2013-11-04 12:00:00,12,8204.7429,397.36590,3,19.647828,19.647828,HIGH,1



Sample LOW anomalies:


,grid_id,timestamp,hour_of_day,total_activity,historical_baseline,baseline_count,deviation,anomaly_score,anomaly_direction,anomaly_flag
145918,869,2013-11-04 13:00:00,13,7.5412,315.69600,3,-0.976112,0.976112,LOW,1
146086,870,2013-11-04 13:00:00,13,7.5412,315.69600,3,-0.976112,0.976112,LOW,1
1362240,8109,2013-11-05 14:00:00,14,8.8228,363.24490,4,-0.975711,0.975711,LOW,1
162718,969,2013-11-04 13:00:00,13,7.8065,315.59600,3,-0.975264,0.975264,LOW,1
507113,3019,2013-11-06 12:00:00,12,14.1806,554.33110,5,-0.974419,0.974419,LOW,1
1362072,8108,2013-11-05 14:00:00,14,9.6593,363.83775,4,-0.973452,0.973452,LOW,1
490313,2919,2013-11-06 12:00:00,12,16.5587,548.31640,5,-0.969801,0.969801,LOW,1
1362226,8109,2013-11-05 12:00:00,12,14.8709,481.75335,4,-0.969132,0.969132,LOW,1
1362058,8108,2013-11-05 12:00:00,12,15.0859,482.05745,4,-0.968705,0.968705,LOW,1
507127,3019,2013-11-06 14:00:00,14,14.1806,447.01900,5,-0.968277,0.968277,LOW,1


In [16]:
# ============================================================
# ML4 - CELL 8
# Human-Readable Anomaly Reason
# ============================================================

logger.info("=" * 70)
logger.info("ML4 CELL 8 - Generating human-readable anomaly reasons")
logger.info("=" * 70)

# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

def generate_anomaly_reason(row):
    """
    Generate a deterministic human-readable explanation
    for an ML4 anomaly classification.
    """

    direction = row["anomaly_direction"]

    grid_id = int(row["grid_id"])
    hour = int(row["hour_of_day"])

    baseline_count = int(row["baseline_count"])

    if direction == HIGH_LABEL:

        deviation_pct = row["deviation"] * 100
        baseline = row["historical_baseline"]

        return (
            f"Activity is {deviation_pct:.1f}% above the "
            f"historical baseline for grid {grid_id} "
            f"at hour {hour:02d} "
            f"(baseline={baseline:.2f}, "
            f"history_count={baseline_count})."
        )

    elif direction == LOW_LABEL:

        deviation_pct = abs(row["deviation"]) * 100
        baseline = row["historical_baseline"]

        return (
            f"Activity is {deviation_pct:.1f}% below the "
            f"historical baseline for grid {grid_id} "
            f"at hour {hour:02d} "
            f"(baseline={baseline:.2f}, "
            f"history_count={baseline_count})."
        )

    elif direction == NORMAL_LABEL:

        deviation_pct = abs(row["deviation"]) * 100

        return (
            f"Activity is within the historical range for "
            f"grid {grid_id} at hour {hour:02d} "
            f"({deviation_pct:.1f}% absolute deviation)."
        )

    else:

        return (
            f"Insufficient historical observations to "
            f"establish a reliable baseline for grid "
            f"{grid_id} at hour {hour:02d} "
            f"(history_count={baseline_count}; "
            f"minimum_required={MIN_HISTORY_COUNT})."
        )


# ------------------------------------------------------------
# Generate reasons
# ------------------------------------------------------------

ml4_activity_df["reason"] = (
    ml4_activity_df.apply(
        generate_anomaly_reason,
        axis=1
    )
)

# ------------------------------------------------------------
# Operational action
# ------------------------------------------------------------

ml4_activity_df["recommended_action"] = np.where(
    ml4_activity_df["anomaly_flag"] == 1,
    ANOMALY_ACTION,
    "NONE"
)

# ------------------------------------------------------------
# Validate reasons
# ------------------------------------------------------------

missing_reasons = (
    ml4_activity_df["reason"]
    .isna()
    .sum()
)

empty_reasons = (
    ml4_activity_df["reason"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

missing_actions = (
    ml4_activity_df["recommended_action"]
    .isna()
    .sum()
)

assert missing_reasons == 0, (
    "Some ML4 rows have missing reasons."
)

assert empty_reasons == 0, (
    "Some ML4 rows have empty reasons."
)

assert missing_actions == 0, (
    "Some ML4 rows have missing recommended actions."
)

# ------------------------------------------------------------
# Validate anomaly actions
# ------------------------------------------------------------

anomaly_action_errors = (
    ml4_activity_df.loc[
        ml4_activity_df["anomaly_flag"] == 1,
        "recommended_action"
    ]
    != ANOMALY_ACTION
).sum()

assert anomaly_action_errors == 0, (
    "Anomaly rows do not consistently have INVESTIGATE action."
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\nML4 human-readable reason generation")
print("-" * 70)

print(
    f"Rows with reasons          : "
    f"{ml4_activity_df['reason'].notna().sum():,}"
)

print(
    f"Missing reasons            : "
    f"{missing_reasons:,}"
)

print(
    f"Empty reasons              : "
    f"{empty_reasons:,}"
)

print(
    f"Missing actions            : "
    f"{missing_actions:,}"
)

print(
    f"Anomaly action errors      : "
    f"{anomaly_action_errors:,}"
)

# ------------------------------------------------------------
# Sample HIGH anomalies
# ------------------------------------------------------------

print("\nSample HIGH anomaly explanations:")
display(
    ml4_activity_df[
        ml4_activity_df["anomaly_direction"] == HIGH_LABEL
    ][
        [
            "grid_id",
            "timestamp",
            "total_activity",
            "historical_baseline",
            "baseline_count",
            "deviation",
            "anomaly_score",
            "anomaly_direction",
            "anomaly_flag",
            "recommended_action",
            "reason"
        ]
    ]
    .sort_values(
        "anomaly_score",
        ascending=False
    )
    .head(10)
)

# ------------------------------------------------------------
# Sample LOW anomalies
# ------------------------------------------------------------

print("\nSample LOW anomaly explanations:")
display(
    ml4_activity_df[
        ml4_activity_df["anomaly_direction"] == LOW_LABEL
    ][
        [
            "grid_id",
            "timestamp",
            "total_activity",
            "historical_baseline",
            "baseline_count",
            "deviation",
            "anomaly_score",
            "anomaly_direction",
            "anomaly_flag",
            "recommended_action",
            "reason"
        ]
    ]
    .sort_values(
        "anomaly_score",
        ascending=False
    )
    .head(10)
)

# ------------------------------------------------------------
# Sample UNKNOWN rows
# ------------------------------------------------------------

print("\nSample UNKNOWN rows:")
display(
    ml4_activity_df[
        ml4_activity_df["anomaly_direction"] == "UNKNOWN"
    ][
        [
            "grid_id",
            "timestamp",
            "total_activity",
            "historical_baseline",
            "baseline_count",
            "anomaly_direction",
            "anomaly_flag",
            "recommended_action",
            "reason"
        ]
    ]
    .head(5)
)

logger.info(
    "Human-readable anomaly reasons generated successfully."
)

logger.info(
    f"Anomaly rows requiring investigation: "
    f"{ml4_activity_df['anomaly_flag'].sum():,}"
)

logger.info(
    "ML4 CELL 8 completed successfully."
)

2026-09-05 16:56:37,871 | INFO | ======================================================================
2026-09-05 16:56:37,872 | INFO | ML4 CELL 8 - Generating human-readable anomaly reasons
2026-09-05 16:56:37,872 | INFO | ======================================================================



ML4 human-readable reason generation
----------------------------------------------------------------------
Rows with reasons          : 1,679,994
Missing reasons            : 0
Empty reasons              : 0
Missing actions            : 0
Anomaly action errors      : 0

Sample HIGH anomaly explanations:


,grid_id,timestamp,total_activity,historical_baseline,baseline_count,deviation,anomaly_score,anomaly_direction,anomaly_flag,recommended_action,reason
768316,4574,2013-11-04 07:00:00,311.8856,12.05700,3,24.867596,24.867596,HIGH,1,INVESTIGATE,Activity is 2486.8% above the historical basel...
768309,4574,2013-11-04 06:00:00,291.6454,11.58220,3,24.180484,24.180484,HIGH,1,INVESTIGATE,Activity is 2418.0% above the historical basel...
1184643,7052,2013-11-05 11:00:00,9631.5100,411.12445,4,22.427237,22.427237,HIGH,1,INVESTIGATE,Activity is 2242.7% above the historical basel...
1297561,7724,2013-11-06 14:00:00,15322.9761,662.46070,5,22.130393,22.130393,HIGH,1,INVESTIGATE,Activity is 2213.0% above the historical basel...
1184656,7052,2013-11-04 13:00:00,7938.9004,353.19810,3,21.477189,21.477189,HIGH,1,INVESTIGATE,Activity is 2147.7% above the historical basel...
1184635,7052,2013-11-04 10:00:00,7752.0294,359.27060,3,20.577133,20.577133,HIGH,1,INVESTIGATE,Activity is 2057.7% above the historical basel...
1184663,7052,2013-11-04 14:00:00,7923.4171,370.44980,3,20.388639,20.388639,HIGH,1,INVESTIGATE,Activity is 2038.9% above the historical basel...
1297554,7724,2013-11-06 13:00:00,14731.7364,694.76140,5,20.204023,20.204023,HIGH,1,INVESTIGATE,Activity is 2020.4% above the historical basel...
768302,4574,2013-11-04 05:00:00,277.8204,13.10530,3,20.199087,20.199087,HIGH,1,INVESTIGATE,Activity is 2019.9% above the historical basel...
1184649,7052,2013-11-04 12:00:00,8204.7429,397.36590,3,19.647828,19.647828,HIGH,1,INVESTIGATE,Activity is 1964.8% above the historical basel...



Sample LOW anomaly explanations:


,grid_id,timestamp,total_activity,historical_baseline,baseline_count,deviation,anomaly_score,anomaly_direction,anomaly_flag,recommended_action,reason
145918,869,2013-11-04 13:00:00,7.5412,315.69600,3,-0.976112,0.976112,LOW,1,INVESTIGATE,Activity is 97.6% below the historical baselin...
146086,870,2013-11-04 13:00:00,7.5412,315.69600,3,-0.976112,0.976112,LOW,1,INVESTIGATE,Activity is 97.6% below the historical baselin...
1362240,8109,2013-11-05 14:00:00,8.8228,363.24490,4,-0.975711,0.975711,LOW,1,INVESTIGATE,Activity is 97.6% below the historical baselin...
162718,969,2013-11-04 13:00:00,7.8065,315.59600,3,-0.975264,0.975264,LOW,1,INVESTIGATE,Activity is 97.5% below the historical baselin...
507113,3019,2013-11-06 12:00:00,14.1806,554.33110,5,-0.974419,0.974419,LOW,1,INVESTIGATE,Activity is 97.4% below the historical baselin...
1362072,8108,2013-11-05 14:00:00,9.6593,363.83775,4,-0.973452,0.973452,LOW,1,INVESTIGATE,Activity is 97.3% below the historical baselin...
490313,2919,2013-11-06 12:00:00,16.5587,548.31640,5,-0.969801,0.969801,LOW,1,INVESTIGATE,Activity is 97.0% below the historical baselin...
1362226,8109,2013-11-05 12:00:00,14.8709,481.75335,4,-0.969132,0.969132,LOW,1,INVESTIGATE,Activity is 96.9% below the historical baselin...
1362058,8108,2013-11-05 12:00:00,15.0859,482.05745,4,-0.968705,0.968705,LOW,1,INVESTIGATE,Activity is 96.9% below the historical baselin...
507127,3019,2013-11-06 14:00:00,14.1806,447.01900,5,-0.968277,0.968277,LOW,1,INVESTIGATE,Activity is 96.8% below the historical baselin...



Sample UNKNOWN rows:


,grid_id,timestamp,total_activity,historical_baseline,baseline_count,anomaly_direction,anomaly_flag,recommended_action,reason
0,1,2013-11-01 00:00:00,62.0092,NaN,0,UNKNOWN,0,NONE,Insufficient historical observations to establ...
1,1,2013-11-02 00:00:00,49.9470,NaN,1,UNKNOWN,0,NONE,Insufficient historical observations to establ...
2,1,2013-11-03 00:00:00,57.4846,NaN,2,UNKNOWN,0,NONE,Insufficient historical observations to establ...
7,1,2013-11-01 01:00:00,46.3654,NaN,0,UNKNOWN,0,NONE,Insufficient historical observations to establ...
8,1,2013-11-02 01:00:00,45.9912,NaN,1,UNKNOWN,0,NONE,Insufficient historical observations to establ...


2026-09-05 16:56:52,135 | INFO | Human-readable anomaly reasons generated successfully.
2026-09-05 16:56:52,137 | INFO | Anomaly rows requiring investigation: 182,209
2026-09-05 16:56:52,138 | INFO | ML4 CELL 8 completed successfully.


In [18]:
# ============================================================
# ML4 - CELL 9A
# Check Saved ML3 Artifacts
# ============================================================

logger.info("=" * 70)
logger.info("ML4 CELL 9A - Checking saved ML3 artifacts")
logger.info("=" * 70)

import os

ML3_DATASET_PATH = "data/ml3_final_engineered_dataset.csv"
ML3_MODEL_PATH = "models/ml3_high_activity_decision_tree.joblib"
ML3_METADATA_PATH = "models/ml3_model_metadata.json"

print("\nSaved ML3 artifacts")
print("-" * 70)

artifacts = {
    "ML3 engineered dataset": ML3_DATASET_PATH,
    "ML3 decision tree": ML3_MODEL_PATH,
    "ML3 metadata": ML3_METADATA_PATH
}

for name, path in artifacts.items():

    exists = os.path.exists(path)

    print(
        f"{name:28s}: "
        f"{'FOUND' if exists else 'NOT FOUND'}"
        f" → {path}"
    )

    if exists:
        size_mb = os.path.getsize(path) / (1024 ** 2)

        print(
            f"{'':28s}  Size: {size_mb:.2f} MB"
        )

logger.info("ML3 artifact existence check completed.")

# ------------------------------------------------------------
# Explicit status
# ------------------------------------------------------------

missing_artifacts = [
    name
    for name, path in artifacts.items()
    if not os.path.exists(path)
]

if missing_artifacts:

    logger.warning(
        f"Missing ML3 artifacts: {missing_artifacts}"
    )

    print("\nWARNING")
    print("-" * 70)
    print(
        "The following ML3 artifacts are missing:"
    )

    for item in missing_artifacts:
        print(f" - {item}")

else:

    logger.info(
        "All required ML3 artifacts are available."
    )

    print("\nAll required ML3 artifacts are available.")

2026-09-05 16:58:53,798 | INFO | ======================================================================
2026-09-05 16:58:53,799 | INFO | ML4 CELL 9A - Checking saved ML3 artifacts
2026-09-05 16:58:53,799 | INFO | ======================================================================
2026-09-05 16:58:53,801 | INFO | ML3 artifact existence check completed.
2026-09-05 16:58:53,801 | INFO | All required ML3 artifacts are available.



Saved ML3 artifacts
----------------------------------------------------------------------
ML3 engineered dataset      : FOUND → data/ml3_final_engineered_dataset.csv
                              Size: 200.20 MB
ML3 decision tree           : FOUND → models/ml3_high_activity_decision_tree.joblib
                              Size: 0.01 MB
ML3 metadata                : FOUND → models/ml3_model_metadata.json
                              Size: 0.01 MB

All required ML3 artifacts are available.


In [21]:
# ============================================================
# ML4 CELL 9B-DIAGNOSTIC — Inspect ML3 timestamp values
# ============================================================

import pandas as pd
import numpy as np

ML3_DATASET_PATH = "data/ml3_final_engineered_dataset.csv"

print("Loading ML3 CSV...")
ml3_raw_df = pd.read_csv(
    ML3_DATASET_PATH,
    usecols=["feature_timestamp"]
)

print(f"Rows loaded: {len(ml3_raw_df):,}")
print(f"Original dtype: {ml3_raw_df['feature_timestamp'].dtype}")

# Show raw examples
print("\nFirst 10 raw timestamp values:")
print(ml3_raw_df["feature_timestamp"].head(10).to_string(index=False))

# Try parsing without modifying the original column
parsed_timestamp = pd.to_datetime(
    ml3_raw_df["feature_timestamp"],
    errors="coerce"
)

invalid_mask = parsed_timestamp.isna()

print("\nParsing result:")
print(f"Valid timestamps  : {(~invalid_mask).sum():,}")
print(f"Invalid timestamps: {invalid_mask.sum():,}")

print("\nExamples of invalid raw timestamp values:")
print(
    ml3_raw_df.loc[
        invalid_mask,
        "feature_timestamp"
    ]
    .head(30)
    .to_string(index=False)
)

print("\nUnique invalid values:")
print(
    ml3_raw_df.loc[
        invalid_mask,
        "feature_timestamp"
    ]
    .drop_duplicates()
    .head(50)
    .to_string(index=False)
)

Loading ML3 CSV...
Rows loaded: 1,199,884
Original dtype: str

First 10 raw timestamp values:
2013-11-02 23:00:00
2013-11-02 23:00:00
2013-11-02 23:00:00
2013-11-02 23:00:00
2013-11-02 23:00:00
2013-11-02 23:00:00
2013-11-02 23:00:00
2013-11-02 23:00:00
2013-11-02 23:00:00
2013-11-02 23:00:00

Parsing result:
Valid timestamps  : 1,185,600
Invalid timestamps: 14,284

Examples of invalid raw timestamp values:
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05
2013-11-05

Unique invalid values:
2013-11-05
2013-11-07


In [22]:
# ============================================================
# ML4 — VERIFY RESAVED ML3 DATASET
# ============================================================

import pandas as pd

ML3_DATASET_PATH = "data/ml3_final_engineered_dataset.csv"

print("Reloading resaved ML3 dataset...")

test_ml3_df = pd.read_csv(
    ML3_DATASET_PATH,
    parse_dates=["feature_timestamp"]
)

print(f"Rows: {len(test_ml3_df):,}")
print(f"Timestamp dtype: {test_ml3_df['feature_timestamp'].dtype}")

invalid_timestamps = test_ml3_df["feature_timestamp"].isna().sum()

print(f"Invalid timestamps: {invalid_timestamps:,}")

print("\nTimestamp range:")
print(
    test_ml3_df["feature_timestamp"].min(),
    "to",
    test_ml3_df["feature_timestamp"].max()
)

print("\nDate/time examples:")
print(
    test_ml3_df["feature_timestamp"]
    .drop_duplicates()
    .sort_values()
    .head(10)
    .to_string(index=False)
)

if invalid_timestamps != 0:
    raise ValueError(
        f"Still found {invalid_timestamps} invalid timestamps."
    )

print("\nSUCCESS — ML3 timestamps are now correctly stored and reloadable.")

Reloading resaved ML3 dataset...
Rows: 1,199,884
Timestamp dtype: datetime64[us]
Invalid timestamps: 0

Timestamp range:
2013-11-02 23:00:00 to 2013-11-07 22:00:00

Date/time examples:
2013-11-02 23:00:00
2013-11-03 00:00:00
2013-11-03 01:00:00
2013-11-03 02:00:00
2013-11-03 03:00:00
2013-11-03 04:00:00
2013-11-03 05:00:00
2013-11-03 06:00:00
2013-11-03 07:00:00
2013-11-03 08:00:00

SUCCESS — ML3 timestamps are now correctly stored and reloadable.


In [24]:
# ============================================================
# ML4 CELL 9B — Reload ML3 artifacts and regenerate predictions
# ============================================================

import os
import json
import joblib
import pandas as pd
import numpy as np

logger = logging.getLogger("ML4")

ML3_DATASET_PATH = "data/ml3_final_engineered_dataset.csv"
ML3_MODEL_PATH = "models/ml3_high_activity_decision_tree.joblib"
ML3_METADATA_PATH = "models/ml3_model_metadata.json"

logger.info("Reloading saved ML3 artifacts for ML4 comparison...")

# ------------------------------------------------------------
# 1. Load ML3 dataset
# ------------------------------------------------------------

ml3_full_df = pd.read_csv(
    ML3_DATASET_PATH,
    parse_dates=["feature_timestamp"]
)

logger.info(
    "ML3 dataset loaded: rows=%d, columns=%d",
    len(ml3_full_df),
    len(ml3_full_df.columns)
)

# ------------------------------------------------------------
# 2. Explicit timestamp validation/conversion
# ------------------------------------------------------------

ml3_full_df["feature_timestamp"] = pd.to_datetime(
    ml3_full_df["feature_timestamp"],
    errors="coerce"
)

if ml3_full_df["feature_timestamp"].isna().any():
    bad_count = ml3_full_df["feature_timestamp"].isna().sum()
    raise ValueError(
        f"ML3 feature_timestamp contains {bad_count} invalid timestamps."
    )

logger.info(
    "feature_timestamp dtype after conversion: %s",
    ml3_full_df["feature_timestamp"].dtype
)

# ------------------------------------------------------------
# 3. Load saved ML3 model
# ------------------------------------------------------------

ml3_model = joblib.load(ML3_MODEL_PATH)

logger.info(
    "ML3 model loaded successfully: %s",
    type(ml3_model).__name__
)

# ------------------------------------------------------------
# 4. ML3 feature list
# ------------------------------------------------------------

ML3_FEATURES = [
    "avg_activity",
    "activity_growth",
    "peak_ratio",
    "variability",
    "internet_share",
    "trailing_median_24h",
    "hour_of_day",
    "day_of_week"
]

missing_features = [
    col for col in ML3_FEATURES
    if col not in ml3_full_df.columns
]

if missing_features:
    raise ValueError(
        f"Missing ML3 model features: {missing_features}"
    )

logger.info("All ML3 model features are present.")

# ------------------------------------------------------------
# 5. Recreate the exact ML3 test period
# ------------------------------------------------------------

TEST_START = pd.Timestamp("2013-11-06 23:00:00")
TEST_END = pd.Timestamp("2013-11-07 22:00:00")

logger.info(
    "ML3 test period: %s to %s",
    TEST_START,
    TEST_END
)

ml3_test_df = ml3_full_df[
    (ml3_full_df["feature_timestamp"] >= TEST_START)
    & (ml3_full_df["feature_timestamp"] <= TEST_END)
].copy()

logger.info(
    "ML3 test rows recovered: %d",
    len(ml3_test_df)
)

# ------------------------------------------------------------
# 6. Validate expected test size
# ------------------------------------------------------------

expected_test_rows = 239_976

if len(ml3_test_df) != expected_test_rows:
    raise ValueError(
        f"Unexpected ML3 test size: {len(ml3_test_df)} "
        f"(expected {expected_test_rows})"
    )

logger.info("ML3 test row count validated successfully.")

# ------------------------------------------------------------
# 7. Generate ML3 predictions
# ------------------------------------------------------------

X_test = ml3_test_df[ML3_FEATURES]

ml3_test_df["ml_prediction_probability"] = (
    ml3_model.predict_proba(X_test)[:, 1]
)

ml3_test_df["ml_prediction"] = (
    ml3_model.predict(X_test)
    .astype(int)
)

logger.info(
    "ML3 predictions generated: positive=%d, negative=%d",
    int((ml3_test_df["ml_prediction"] == 1).sum()),
    int((ml3_test_df["ml_prediction"] == 0).sum())
)

# ------------------------------------------------------------
# 8. Reconstruct target timestamp
# ------------------------------------------------------------

ml3_test_df["target_timestamp"] = (
    ml3_test_df["feature_timestamp"]
    + pd.Timedelta(hours=1)
)

# ------------------------------------------------------------
# 9. Build comparison dataframe
# ------------------------------------------------------------

ml3_compare_df = ml3_test_df[
    [
        "grid_id",
        "feature_timestamp",
        "target_timestamp",
        "target_activity",
        "high_activity",
        "ml_prediction_probability",
        "ml_prediction"
    ]
].copy()

# ML4 / NP3 comparisons will align on target timestamp
ml3_compare_df["timestamp"] = (
    ml3_compare_df["target_timestamp"]
)

# ------------------------------------------------------------
# 10. Validate prediction alignment
# ------------------------------------------------------------

timestamp_difference = (
    ml3_compare_df["target_timestamp"]
    - ml3_compare_df["feature_timestamp"]
)

invalid_target_alignment = (
    timestamp_difference != pd.Timedelta(hours=1)
).sum()

if invalid_target_alignment != 0:
    raise ValueError(
        f"Found {invalid_target_alignment} invalid ML3 target alignments."
    )

duplicate_count = ml3_compare_df.duplicated(
    subset=["grid_id", "timestamp"]
).sum()

if duplicate_count != 0:
    raise ValueError(
        f"Found {duplicate_count} duplicate ML3 grid/timestamp rows."
    )

logger.info(
    "ML3 alignment validation passed: %d rows.",
    len(ml3_compare_df)
)

logger.info(
    "ML3 comparison dataframe ready: %d rows, %d grids.",
    len(ml3_compare_df),
    ml3_compare_df["grid_id"].nunique()
)

print("\nML3 comparison dataset ready.")
print(f"Rows       : {len(ml3_compare_df):,}")
print(f"Grids      : {ml3_compare_df['grid_id'].nunique():,}")
print(f"Time range : {ml3_compare_df['timestamp'].min()} "
      f"to {ml3_compare_df['timestamp'].max()}")

print("\nPrediction distribution:")
print(
    ml3_compare_df["ml_prediction"]
    .value_counts()
    .sort_index()
)

2026-09-05 17:14:10,124 | INFO | Reloading saved ML3 artifacts for ML4 comparison...
2026-09-05 17:14:12,053 | INFO | ML3 dataset loaded: rows=1199884, columns=14
2026-09-05 17:14:12,093 | INFO | feature_timestamp dtype after conversion: datetime64[us]
2026-09-05 17:14:12,095 | INFO | ML3 model loaded successfully: DecisionTreeClassifier
2026-09-05 17:14:12,096 | INFO | All ML3 model features are present.
2026-09-05 17:14:12,097 | INFO | ML3 test period: 2013-11-06 23:00:00 to 2013-11-07 22:00:00
2026-09-05 17:14:12,142 | INFO | ML3 test rows recovered: 239976
2026-09-05 17:14:12,144 | INFO | ML3 test row count validated successfully.
2026-09-05 17:14:12,242 | INFO | ML3 predictions generated: positive=45455, negative=194521
2026-09-05 17:14:12,276 | INFO | ML3 alignment validation passed: 239976 rows.
2026-09-05 17:14:12,279 | INFO | ML3 comparison dataframe ready: 239976 rows, 9999 grids.



ML3 comparison dataset ready.
Rows       : 239,976
Grids      : 9,999
Time range : 2013-11-07 00:00:00 to 2013-11-07 23:00:00

Prediction distribution:
ml_prediction
0    194521
1     45455
Name: count, dtype: int64


In [25]:
# ============================================================
# ML4 CELL 10 — ML4 ANOMALY vs ML3 PREDICTION COMPARISON
# ============================================================

logger = logging.getLogger("ML4")

logger.info("Starting ML4 vs ML3 comparison...")

# ------------------------------------------------------------
# 1. Prepare ML4 comparison columns
# ------------------------------------------------------------

ml4_compare_columns = [
    "grid_id",
    "timestamp",
    "total_activity",
    "historical_baseline",
    "baseline_count",
    "deviation",
    "anomaly_score",
    "anomaly_direction",
    "anomaly_flag",
    "reason",
    "recommended_action",
    "hour_of_day",
]

missing_ml4_columns = [
    col for col in ml4_compare_columns
    if col not in ml4_activity_df.columns
]

if missing_ml4_columns:
    raise ValueError(
        f"Missing ML4 columns: {missing_ml4_columns}"
    )

ml4_compare_base = ml4_activity_df[
    ml4_compare_columns
].copy()

# ------------------------------------------------------------
# 2. Ensure timestamp types are compatible
# ------------------------------------------------------------

ml4_compare_base["timestamp"] = pd.to_datetime(
    ml4_compare_base["timestamp"],
    errors="raise"
)

ml3_compare_df["timestamp"] = pd.to_datetime(
    ml3_compare_df["timestamp"],
    errors="raise"
)

logger.info(
    "Timestamp dtypes — ML4: %s | ML3: %s",
    ml4_compare_base["timestamp"].dtype,
    ml3_compare_df["timestamp"].dtype
)

# ------------------------------------------------------------
# 3. Check uniqueness before joining
# ------------------------------------------------------------

ml4_duplicates = ml4_compare_base.duplicated(
    subset=["grid_id", "timestamp"]
).sum()

ml3_duplicates = ml3_compare_df.duplicated(
    subset=["grid_id", "timestamp"]
).sum()

if ml4_duplicates != 0:
    raise ValueError(
        f"ML4 contains {ml4_duplicates} duplicate grid/timestamp rows."
    )

if ml3_duplicates != 0:
    raise ValueError(
        f"ML3 contains {ml3_duplicates} duplicate grid/timestamp rows."
    )

logger.info("ML4 and ML3 uniqueness checks passed.")

# ------------------------------------------------------------
# 4. Inner join on grid + target timestamp
# ------------------------------------------------------------

ml4_ml3_compare_df = ml3_compare_df.merge(
    ml4_compare_base,
    on=["grid_id", "timestamp"],
    how="inner",
    validate="one_to_one",
    suffixes=("_ml3", "_ml4")
)

logger.info(
    "ML4 vs ML3 aligned rows: %d",
    len(ml4_ml3_compare_df)
)

# ------------------------------------------------------------
# 5. Validate expected alignment
# ------------------------------------------------------------

if len(ml4_ml3_compare_df) != len(ml3_compare_df):
    logger.warning(
        "Not all ML3 rows matched ML4. ML3=%d, matched=%d",
        len(ml3_compare_df),
        len(ml4_ml3_compare_df)
    )

# ------------------------------------------------------------
# 6. Create primary agreement categories
# ------------------------------------------------------------

def classify_ml4_ml3(row):
    ml3_positive = row["ml_prediction"] == 1
    ml4_anomaly = row["anomaly_flag"] == 1

    if ml3_positive and ml4_anomaly:
        return "BOTH_ANOMALY"

    if ml3_positive and not ml4_anomaly:
        return "ML3_ONLY"

    if not ml3_positive and ml4_anomaly:
        return "ML4_ONLY"

    return "NEITHER"


ml4_ml3_compare_df["agreement_category"] = (
    ml4_ml3_compare_df.apply(
        classify_ml4_ml3,
        axis=1
    )
)

# ------------------------------------------------------------
# 7. Direction-specific comparison
# ------------------------------------------------------------

def classify_direction(row):
    ml3_positive = row["ml_prediction"] == 1
    ml4_high = row["anomaly_direction"] == "HIGH"
    ml4_low = row["anomaly_direction"] == "LOW"

    if ml3_positive and ml4_high:
        return "BOTH_ML3_AND_ML4_HIGH"

    if ml3_positive and not ml4_high:
        return "ML3_HIGH_ONLY"

    if not ml3_positive and ml4_high:
        return "ML4_HIGH_ONLY"

    if ml4_low:
        return "ML4_LOW"

    return "NEITHER_HIGH"


ml4_ml3_compare_df["direction_category"] = (
    ml4_ml3_compare_df.apply(
        classify_direction,
        axis=1
    )
)

# ------------------------------------------------------------
# 8. Summary counts
# ------------------------------------------------------------

agreement_summary = (
    ml4_ml3_compare_df["agreement_category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="rows")
)

agreement_summary["percentage"] = (
    agreement_summary["rows"]
    / len(ml4_ml3_compare_df)
    * 100
)

agreement_summary = agreement_summary.sort_values(
    "category"
).reset_index(drop=True)

# ------------------------------------------------------------
# 9. Direction summary
# ------------------------------------------------------------

direction_summary = (
    ml4_ml3_compare_df["direction_category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="rows")
)

direction_summary["percentage"] = (
    direction_summary["rows"]
    / len(ml4_ml3_compare_df)
    * 100
)

direction_summary = direction_summary.sort_values(
    "category"
).reset_index(drop=True)

# ------------------------------------------------------------
# 10. Print results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ML4 vs ML3 — PRIMARY ANOMALY AGREEMENT")
print("=" * 70)

print(agreement_summary.to_string(index=False))

print("\n" + "=" * 70)
print("ML4 vs ML3 — DIRECTION-SPECIFIC COMPARISON")
print("=" * 70)

print(direction_summary.to_string(index=False))

print("\n" + "=" * 70)
print("OVERALL COUNTS")
print("=" * 70)

print(f"Aligned rows       : {len(ml4_ml3_compare_df):,}")
print(
    f"ML3 positive       : "
    f"{(ml4_ml3_compare_df['ml_prediction'] == 1).sum():,}"
)
print(
    f"ML4 anomaly        : "
    f"{(ml4_ml3_compare_df['anomaly_flag'] == 1).sum():,}"
)
print(
    f"ML4 HIGH           : "
    f"{(ml4_ml3_compare_df['anomaly_direction'] == 'HIGH').sum():,}"
)
print(
    f"ML4 LOW            : "
    f"{(ml4_ml3_compare_df['anomaly_direction'] == 'LOW').sum():,}"
)
print(
    f"ML4 NORMAL         : "
    f"{(ml4_ml3_compare_df['anomaly_direction'] == 'NORMAL').sum():,}"
)
print(
    f"ML4 UNKNOWN        : "
    f"{(ml4_ml3_compare_df['anomaly_direction'] == 'UNKNOWN').sum():,}"
)

logger.info("ML4 vs ML3 comparison completed successfully.")

2026-09-05 17:16:11,611 | INFO | Starting ML4 vs ML3 comparison...
2026-09-05 17:16:11,804 | INFO | Timestamp dtypes — ML4: datetime64[us] | ML3: datetime64[us]
2026-09-05 17:16:11,920 | INFO | ML4 and ML3 uniqueness checks passed.
2026-09-05 17:16:12,346 | INFO | ML4 vs ML3 aligned rows: 239976
2026-09-05 17:16:15,555 | INFO | ML4 vs ML3 comparison completed successfully.



ML4 vs ML3 — PRIMARY ANOMALY AGREEMENT
    category   rows  percentage
BOTH_ANOMALY   9615    4.006651
    ML3_ONLY  35840   14.934827
    ML4_ONLY   7482    3.117812
     NEITHER 187039   77.940711

ML4 vs ML3 — DIRECTION-SPECIFIC COMPARISON
             category   rows  percentage
BOTH_ML3_AND_ML4_HIGH   8913    3.714121
        ML3_HIGH_ONLY  36542   15.227356
        ML4_HIGH_ONLY   6144    2.560256
              ML4_LOW   1338    0.557556
         NEITHER_HIGH 187039   77.940711

OVERALL COUNTS
Aligned rows       : 239,976
ML3 positive       : 45,455
ML4 anomaly        : 17,097
ML4 HIGH           : 15,057
ML4 LOW            : 2,040
ML4 NORMAL         : 222,879
ML4 UNKNOWN        : 0


In [29]:
# ============================================================
# ML4 CELL 11 — ML3 vs ML4 DISAGREEMENT ANALYSIS
# CLEAN VERSION
# ============================================================

logger = logging.getLogger("ML4")

logger.info("Starting ML3 vs ML4 disagreement analysis...")

# ------------------------------------------------------------
# 1. Build ML3 analysis dataframe from the saved/test dataframe
# ------------------------------------------------------------

ML3_ANALYSIS_COLUMNS = [
    "grid_id",
    "feature_timestamp",
    "target_activity",
    "ml_prediction_probability",
    "ml_prediction",
    "peak_ratio",
    "activity_growth",
    "trailing_median_24h",
]

missing_ml3 = [
    col for col in ML3_ANALYSIS_COLUMNS
    if col not in ml3_test_df.columns
]

if missing_ml3:
    raise ValueError(
        f"Missing columns in ml3_test_df: {missing_ml3}"
    )

ml3_analysis_df = ml3_test_df[
    ML3_ANALYSIS_COLUMNS
].copy()

# Explicit datetime conversion
ml3_analysis_df["feature_timestamp"] = pd.to_datetime(
    ml3_analysis_df["feature_timestamp"],
    errors="raise"
)

# ML3 predicts one hour ahead
ml3_analysis_df["target_timestamp"] = (
    ml3_analysis_df["feature_timestamp"]
    + pd.Timedelta(hours=1)
)

# ------------------------------------------------------------
# 2. Validate ML3 temporal relationship
# ------------------------------------------------------------

invalid_ml3_alignment = (
    (
        ml3_analysis_df["target_timestamp"]
        - ml3_analysis_df["feature_timestamp"]
    )
    != pd.Timedelta(hours=1)
).sum()

print(
    f"Invalid ML3 feature → target alignments: "
    f"{invalid_ml3_alignment:,}"
)

if invalid_ml3_alignment != 0:
    raise ValueError(
        "ML3 feature-to-target temporal alignment failed."
    )

# ------------------------------------------------------------
# 3. Prepare ML4 dataframe
# ------------------------------------------------------------

ML4_COLUMNS = [
    "grid_id",
    "timestamp",
    "total_activity",
    "historical_baseline",
    "baseline_count",
    "deviation",
    "anomaly_score",
    "anomaly_direction",
    "anomaly_flag",
    "reason",
    "recommended_action",
    "hour_of_day",
]

missing_ml4 = [
    col for col in ML4_COLUMNS
    if col not in ml4_activity_df.columns
]

if missing_ml4:
    raise ValueError(
        f"Missing columns in ml4_activity_df: {missing_ml4}"
    )

ml4_analysis_df = ml4_activity_df[
    ML4_COLUMNS
].copy()

ml4_analysis_df["timestamp"] = pd.to_datetime(
    ml4_analysis_df["timestamp"],
    errors="raise"
)

# ------------------------------------------------------------
# 4. Validate uniqueness
# ------------------------------------------------------------

ml3_dup = ml3_analysis_df.duplicated(
    subset=["grid_id", "target_timestamp"]
).sum()

ml4_dup = ml4_analysis_df.duplicated(
    subset=["grid_id", "timestamp"]
).sum()

print(f"ML3 duplicate grid/target rows: {ml3_dup:,}")
print(f"ML4 duplicate grid/timestamp rows: {ml4_dup:,}")

if ml3_dup != 0:
    raise ValueError("ML3 contains duplicate prediction targets.")

if ml4_dup != 0:
    raise ValueError("ML4 contains duplicate grid/timestamp rows.")

# ------------------------------------------------------------
# 5. Rename ML3 target timestamp for clean join
# ------------------------------------------------------------

ml3_analysis_df = ml3_analysis_df.rename(
    columns={
        "target_timestamp": "timestamp"
    }
)

# ------------------------------------------------------------
# 6. Align ML3 prediction with ML4 observation
# ------------------------------------------------------------

ml4_ml3_compare_df = ml3_analysis_df.merge(
    ml4_analysis_df,
    on=["grid_id", "timestamp"],
    how="inner",
    validate="one_to_one",
    suffixes=("_ml3", "_ml4")
)

logger.info(
    "ML3/ML4 aligned rows: %d",
    len(ml4_ml3_compare_df)
)

# ------------------------------------------------------------
# 7. Validate alignment
# ------------------------------------------------------------

if len(ml4_ml3_compare_df) != len(ml3_analysis_df):
    logger.warning(
        "Some ML3 prediction rows did not match ML4: "
        "ML3=%d, matched=%d",
        len(ml3_analysis_df),
        len(ml4_ml3_compare_df)
    )

# ------------------------------------------------------------
# 8. Create HIGH comparison categories
# ------------------------------------------------------------

def high_comparison_group(row):

    ml3_high = row["ml_prediction"] == 1
    ml4_high = row["anomaly_direction"] == "HIGH"

    if ml3_high and ml4_high:
        return "BOTH_HIGH"

    elif ml3_high and not ml4_high:
        return "ML3_HIGH_ONLY"

    elif not ml3_high and ml4_high:
        return "ML4_HIGH_ONLY"

    else:
        return "NEITHER_HIGH"


ml4_ml3_compare_df["high_comparison_group"] = (
    ml4_ml3_compare_df.apply(
        high_comparison_group,
        axis=1
    )
)

# ------------------------------------------------------------
# 9. Create overall anomaly comparison
# ------------------------------------------------------------

def anomaly_comparison_group(row):

    ml3_positive = row["ml_prediction"] == 1
    ml4_anomaly = row["anomaly_flag"] == 1

    if ml3_positive and ml4_anomaly:
        return "BOTH_ANOMALY"

    elif ml3_positive and not ml4_anomaly:
        return "ML3_ONLY"

    elif not ml3_positive and ml4_anomaly:
        return "ML4_ONLY"

    else:
        return "NEITHER"


ml4_ml3_compare_df["anomaly_comparison_group"] = (
    ml4_ml3_compare_df.apply(
        anomaly_comparison_group,
        axis=1
    )
)

# ------------------------------------------------------------
# 10. Group count summary
# ------------------------------------------------------------

high_group_summary = (
    ml4_ml3_compare_df[
        "high_comparison_group"
    ]
    .value_counts()
    .rename_axis("group")
    .reset_index(name="rows")
)

high_group_summary["percentage"] = (
    high_group_summary["rows"]
    / len(ml4_ml3_compare_df)
    * 100
)

high_group_summary = high_group_summary.sort_values(
    "group"
).reset_index(drop=True)

# ------------------------------------------------------------
# 11. Overall anomaly summary
# ------------------------------------------------------------

anomaly_group_summary = (
    ml4_ml3_compare_df[
        "anomaly_comparison_group"
    ]
    .value_counts()
    .rename_axis("group")
    .reset_index(name="rows")
)

anomaly_group_summary["percentage"] = (
    anomaly_group_summary["rows"]
    / len(ml4_ml3_compare_df)
    * 100
)

anomaly_group_summary = anomaly_group_summary.sort_values(
    "group"
).reset_index(drop=True)

# ------------------------------------------------------------
# 12. Characteristics by HIGH comparison group
# ------------------------------------------------------------

ANALYSIS_COLUMNS = [
    "peak_ratio",
    "activity_growth",
    "trailing_median_24h",
    "target_activity",
    "ml_prediction_probability",
    "deviation",
    "historical_baseline",
]

median_summary = (
    ml4_ml3_compare_df
    .groupby("high_comparison_group")[ANALYSIS_COLUMNS]
    .median()
    .round(4)
)

mean_summary = (
    ml4_ml3_compare_df
    .groupby("high_comparison_group")[ANALYSIS_COLUMNS]
    .mean()
    .round(4)
)

# ------------------------------------------------------------
# 13. Hour-of-day distribution
# ------------------------------------------------------------

hour_distribution = pd.crosstab(
    ml4_ml3_compare_df["hour_of_day"],
    ml4_ml3_compare_df["high_comparison_group"]
)

hour_distribution_percentage = (
    hour_distribution
    .div(
        hour_distribution.sum(axis=0),
        axis=1
    )
    * 100
)

# ------------------------------------------------------------
# 14. Print results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ML3 vs ML4 — HIGH ACTIVITY COMPARISON")
print("=" * 80)

print(
    high_group_summary.to_string(index=False)
)

print("\n" + "=" * 80)
print("ML3 vs ML4 — OVERALL ANOMALY COMPARISON")
print("=" * 80)

print(
    anomaly_group_summary.to_string(index=False)
)

print("\n" + "=" * 80)
print("MEDIAN CHARACTERISTICS BY HIGH COMPARISON GROUP")
print("=" * 80)

print(
    median_summary.to_string()
)

print("\n" + "=" * 80)
print("MEAN CHARACTERISTICS BY HIGH COMPARISON GROUP")
print("=" * 80)

print(
    mean_summary.to_string()
)

print("\n" + "=" * 80)
print("HOUR-OF-DAY DISTRIBUTION (%)")
print("=" * 80)

print(
    hour_distribution_percentage
    .round(2)
    .to_string()
)

print("\n" + "=" * 80)
print("FINAL VALIDATION")
print("=" * 80)

print(f"ML3 rows                 : {len(ml3_analysis_df):,}")
print(f"ML4/ML3 aligned rows     : {len(ml4_ml3_compare_df):,}")
print(
    f"ML3 positive predictions : "
    f"{(ml4_ml3_compare_df['ml_prediction'] == 1).sum():,}"
)
print(
    f"ML4 HIGH anomalies       : "
    f"{(ml4_ml3_compare_df['anomaly_direction'] == 'HIGH').sum():,}"
)
print(
    f"ML4 LOW anomalies        : "
    f"{(ml4_ml3_compare_df['anomaly_direction'] == 'LOW').sum():,}"
)

logger.info(
    "ML3 vs ML4 disagreement analysis completed successfully."
)

print("\nML3 vs ML4 disagreement analysis completed successfully.")

2026-09-05 17:20:56,036 | INFO | Starting ML3 vs ML4 disagreement analysis...


Invalid ML3 feature → target alignments: 0
ML3 duplicate grid/target rows: 0
ML4 duplicate grid/timestamp rows: 0


2026-09-05 17:20:56,660 | INFO | ML3/ML4 aligned rows: 239976
2026-09-05 17:20:59,408 | INFO | ML3 vs ML4 disagreement analysis completed successfully.



ML3 vs ML4 — HIGH ACTIVITY COMPARISON
        group   rows  percentage
    BOTH_HIGH   8913    3.714121
ML3_HIGH_ONLY  36542   15.227356
ML4_HIGH_ONLY   6144    2.560256
 NEITHER_HIGH 188377   78.498266

ML3 vs ML4 — OVERALL ANOMALY COMPARISON
       group   rows  percentage
BOTH_ANOMALY   9615    4.006651
    ML3_ONLY  35840   14.934827
    ML4_ONLY   7482    3.117812
     NEITHER 187039   77.940711

MEDIAN CHARACTERISTICS BY HIGH COMPARISON GROUP
                       peak_ratio  activity_growth  trailing_median_24h  target_activity  ml_prediction_probability  deviation  historical_baseline
high_comparison_group                                                                                                                              
BOTH_HIGH                  1.9954           0.0085             323.0071         596.4101                     0.8561     0.6554             348.3306
ML3_HIGH_ONLY              1.8376          -0.0021             222.7418         275.3948              

In [30]:
# ============================================================
# ML4 CELL 12 — LOAD NP3 ALERTS FOR COMPARISON
# ============================================================

import pandas as pd
import logging

logger = logging.getLogger("ML4")

logger.info("Loading NP3 alerts for ML4 comparison...")

# ------------------------------------------------------------
# MySQL connection
# ------------------------------------------------------------

# Use your existing MySQL connection if already available.
# If your notebook already has `conn`, this will reuse it.

try:
    conn
    logger.info("Using existing MySQL connection.")
except NameError:
    raise NameError(
        "MySQL connection `conn` is not available. "
        "Please use the same connection object from your NP3 notebook/cells."
    )

# ------------------------------------------------------------
# Load only NP3 alert types relevant to high-activity analysis
# ------------------------------------------------------------

np3_query = """
SELECT
    id,
    grid_id,
    timestamp,
    alert_type,
    severity,
    current_activity,
    baseline_activity,
    reason,
    created_at
FROM activity_alerts
WHERE alert_type IN ('HIGH_ACTIVITY', 'ACTIVITY_SPIKE')
"""

np3_alerts_df = pd.read_sql(
    np3_query,
    conn
)

# ------------------------------------------------------------
# Timestamp normalization
# ------------------------------------------------------------

np3_alerts_df["timestamp"] = pd.to_datetime(
    np3_alerts_df["timestamp"],
    errors="raise"
)

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NP3 ALERT DATA")
print("=" * 80)

print(f"Rows                 : {len(np3_alerts_df):,}")
print(f"Unique grids         : {np3_alerts_df['grid_id'].nunique():,}")
print(
    f"Timestamp range      : "
    f"{np3_alerts_df['timestamp'].min()} "
    f"to "
    f"{np3_alerts_df['timestamp'].max()}"
)

print("\nAlert type counts:")
print(
    np3_alerts_df["alert_type"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# Duplicate validation
# ------------------------------------------------------------

duplicate_alerts = np3_alerts_df.duplicated(
    subset=["grid_id", "timestamp", "alert_type"]
).sum()

print(
    f"\nDuplicate grid/timestamp/alert_type rows: "
    f"{duplicate_alerts:,}"
)

if duplicate_alerts != 0:
    raise ValueError(
        f"Found {duplicate_alerts} duplicate NP3 alerts."
    )

# ------------------------------------------------------------
# Check required columns
# ------------------------------------------------------------

required_np3_columns = [
    "grid_id",
    "timestamp",
    "alert_type",
    "current_activity",
    "baseline_activity",
    "reason",
]

missing_np3_columns = [
    col
    for col in required_np3_columns
    if col not in np3_alerts_df.columns
]

if missing_np3_columns:
    raise ValueError(
        f"Missing NP3 columns: {missing_np3_columns}"
    )

logger.info(
    "NP3 alerts loaded successfully: %d rows.",
    len(np3_alerts_df)
)

print("\nNP3 alert loading and validation completed successfully.")

2026-09-05 17:22:54,139 | INFO | Loading NP3 alerts for ML4 comparison...


NameError: MySQL connection `conn` is not available. Please use the same connection object from your NP3 notebook/cells.

In [31]:
# ============================================================
# ML4 — CHECK AVAILABLE DATABASE CONNECTION VARIABLES
# ============================================================

print("Checking existing notebook variables related to MySQL...")

possible_vars = [
    "conn",
    "connection",
    "mysql_conn",
    "db_conn",
    "engine",
    "mysql_engine",
    "jdbc_url",
    "MYSQL_HOST",
    "MYSQL_PORT",
    "MYSQL_DATABASE",
    "DB_HOST",
    "DB_NAME",
]

found = []

for name in possible_vars:
    if name in globals():
        found.append(name)

print("\nVariables found:")
if found:
    for name in found:
        print(f"  - {name}")
else:
    print("  None of the common MySQL connection variables were found.")

print("\nML4 notebook database-related variables:")
for name in sorted(globals()):
    if any(term in name.lower() for term in ["mysql", "db_", "database", "jdbc", "engine", "conn"]):
        print(f"  - {name}")

Checking existing notebook variables related to MySQL...

Variables found:
  - engine
  - MYSQL_HOST
  - MYSQL_PORT
  - MYSQL_DATABASE

ML4 notebook database-related variables:
  - MYSQL_DATABASE
  - MYSQL_HOST
  - MYSQL_PASSWORD
  - MYSQL_PORT
  - MYSQL_URL
  - MYSQL_USER
  - create_engine
  - engine


In [32]:
# ============================================================
# ML4 CELL 12 — LOAD NP3 ALERTS FOR COMPARISON
# ============================================================

import pandas as pd
import logging

logger = logging.getLogger("ML4")

logger.info("Loading NP3 alerts from activity_alerts...")

# ------------------------------------------------------------
# 1. Verify existing SQLAlchemy engine
# ------------------------------------------------------------

if "engine" not in globals():
    raise NameError(
        "Existing SQLAlchemy `engine` was not found."
    )

logger.info("Using existing SQLAlchemy MySQL engine.")

# ------------------------------------------------------------
# 2. Load NP3 alert records
# ------------------------------------------------------------

np3_query = """
SELECT
    id,
    grid_id,
    timestamp,
    alert_type,
    severity,
    current_activity,
    baseline_activity,
    reason,
    created_at
FROM activity_alerts
WHERE alert_type IN ('HIGH_ACTIVITY', 'ACTIVITY_SPIKE')
"""

np3_alerts_df = pd.read_sql(
    np3_query,
    engine
)

# ------------------------------------------------------------
# 3. Normalize timestamp
# ------------------------------------------------------------

np3_alerts_df["timestamp"] = pd.to_datetime(
    np3_alerts_df["timestamp"],
    errors="raise"
)

# ------------------------------------------------------------
# 4. Validate required columns
# ------------------------------------------------------------

required_np3_columns = [
    "grid_id",
    "timestamp",
    "alert_type",
    "severity",
    "current_activity",
    "baseline_activity",
    "reason",
]

missing_np3_columns = [
    col
    for col in required_np3_columns
    if col not in np3_alerts_df.columns
]

if missing_np3_columns:
    raise ValueError(
        f"Missing NP3 columns: {missing_np3_columns}"
    )

# ------------------------------------------------------------
# 5. Validate duplicates
# ------------------------------------------------------------

duplicate_alerts = np3_alerts_df.duplicated(
    subset=["grid_id", "timestamp", "alert_type"]
).sum()

if duplicate_alerts != 0:
    raise ValueError(
        f"Found {duplicate_alerts} duplicate "
        "NP3 grid/timestamp/alert_type records."
    )

# ------------------------------------------------------------
# 6. Print NP3 summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NP3 ALERT DATA")
print("=" * 80)

print(f"Rows            : {len(np3_alerts_df):,}")
print(f"Unique grids    : {np3_alerts_df['grid_id'].nunique():,}")
print(
    f"Timestamp range : "
    f"{np3_alerts_df['timestamp'].min()} "
    f"to "
    f"{np3_alerts_df['timestamp'].max()}"
)

print("\nAlert type counts:")
print(
    np3_alerts_df["alert_type"]
    .value_counts()
    .sort_index()
    .to_string()
)

print(
    f"\nDuplicate grid/timestamp/alert_type rows: "
    f"{duplicate_alerts:,}"
)

print("\nSeverity counts:")
print(
    np3_alerts_df["severity"]
    .value_counts()
    .sort_index()
    .to_string()
)

logger.info(
    "NP3 alerts loaded successfully: %d rows.",
    len(np3_alerts_df)
)

print("\nNP3 alert loading and validation completed successfully.")

2026-09-05 17:25:40,114 | INFO | Loading NP3 alerts from activity_alerts...
2026-09-05 17:25:40,116 | INFO | Using existing SQLAlchemy MySQL engine.
2026-09-05 17:25:44,146 | INFO | NP3 alerts loaded successfully: 188812 rows.



NP3 ALERT DATA
Rows            : 188,812
Unique grids    : 9,064
Timestamp range : 2013-11-01 00:00:00 to 2013-11-07 23:00:00

Alert type counts:
alert_type
ACTIVITY_SPIKE     61948
HIGH_ACTIVITY     126864

Duplicate grid/timestamp/alert_type rows: 0

Severity counts:
severity
HIGH    188812

NP3 alert loading and validation completed successfully.


In [33]:
# ============================================================
# ML4 CELL 13 — ML4 vs NP3 COMPARISON
# ============================================================

logger = logging.getLogger("ML4")

logger.info("Starting ML4 vs NP3 comparison...")

# ------------------------------------------------------------
# 1. Restrict ML4 to the ML3 test/comparison period
# ------------------------------------------------------------

ml4_test_df = ml4_activity_df[
    ml4_activity_df["timestamp"].between(
        ml3_compare_df["timestamp"].min(),
        ml3_compare_df["timestamp"].max()
    )
].copy()

logger.info(
    "ML4 rows in comparison period: %d",
    len(ml4_test_df)
)

# ------------------------------------------------------------
# 2. Build NP3 HIGH_ACTIVITY indicator
# ------------------------------------------------------------

np3_high_df = np3_alerts_df[
    np3_alerts_df["alert_type"] == "HIGH_ACTIVITY"
][
    ["grid_id", "timestamp", "current_activity",
     "baseline_activity", "reason"]
].copy()

np3_high_df["np3_high_activity"] = 1

np3_high_df = np3_high_df.rename(
    columns={
        "current_activity": "np3_current_activity",
        "baseline_activity": "np3_baseline_activity",
        "reason": "np3_high_reason"
    }
)

# ------------------------------------------------------------
# 3. Build NP3 ACTIVITY_SPIKE indicator
# ------------------------------------------------------------

np3_spike_df = np3_alerts_df[
    np3_alerts_df["alert_type"] == "ACTIVITY_SPIKE"
][
    ["grid_id", "timestamp", "current_activity",
     "baseline_activity", "reason"]
].copy()

np3_spike_df["np3_activity_spike"] = 1

np3_spike_df = np3_spike_df.rename(
    columns={
        "current_activity": "np3_spike_current_activity",
        "baseline_activity": "np3_spike_baseline_activity",
        "reason": "np3_spike_reason"
    }
)

# ------------------------------------------------------------
# 4. Create complete ML4 comparison base
# ------------------------------------------------------------

ml4_np3_compare_df = ml4_test_df[
    [
        "grid_id",
        "timestamp",
        "total_activity",
        "historical_baseline",
        "baseline_count",
        "deviation",
        "anomaly_score",
        "anomaly_direction",
        "anomaly_flag",
        "reason",
        "recommended_action",
        "hour_of_day"
    ]
].copy()

# ------------------------------------------------------------
# 5. Merge HIGH_ACTIVITY alerts
# ------------------------------------------------------------

ml4_np3_compare_df = ml4_np3_compare_df.merge(
    np3_high_df[
        [
            "grid_id",
            "timestamp",
            "np3_high_activity",
            "np3_current_activity",
            "np3_baseline_activity",
            "np3_high_reason"
        ]
    ],
    on=["grid_id", "timestamp"],
    how="left",
    validate="one_to_one"
)

# Missing alert = no recorded NP3 HIGH_ACTIVITY alert
ml4_np3_compare_df["np3_high_activity"] = (
    ml4_np3_compare_df["np3_high_activity"]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 6. Merge ACTIVITY_SPIKE alerts
# ------------------------------------------------------------

ml4_np3_compare_df = ml4_np3_compare_df.merge(
    np3_spike_df[
        [
            "grid_id",
            "timestamp",
            "np3_activity_spike",
            "np3_spike_current_activity",
            "np3_spike_baseline_activity",
            "np3_spike_reason"
        ]
    ],
    on=["grid_id", "timestamp"],
    how="left",
    validate="one_to_one"
)

ml4_np3_compare_df["np3_activity_spike"] = (
    ml4_np3_compare_df["np3_activity_spike"]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 7. Validate comparison rows
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ML4 / NP3 ALIGNMENT")
print("=" * 80)

print(f"ML4 comparison rows: {len(ml4_np3_compare_df):,}")

print(
    f"NP3 HIGH_ACTIVITY matches: "
    f"{ml4_np3_compare_df['np3_high_activity'].sum():,}"
)

print(
    f"NP3 ACTIVITY_SPIKE matches: "
    f"{ml4_np3_compare_df['np3_activity_spike'].sum():,}"
)

# ------------------------------------------------------------
# 8. Generic agreement function
# ------------------------------------------------------------

def agreement_category(ml4_flag, np3_flag):

    if ml4_flag == 1 and np3_flag == 1:
        return "BOTH_AGREE"

    elif ml4_flag == 1 and np3_flag == 0:
        return "ML4_ONLY"

    elif ml4_flag == 0 and np3_flag == 1:
        return "NP3_ONLY"

    return "NEITHER"

# ------------------------------------------------------------
# 9. ML4 anomaly vs NP3 HIGH_ACTIVITY
# ------------------------------------------------------------

ml4_np3_compare_df["ml4_vs_np3_high"] = (
    ml4_np3_compare_df.apply(
        lambda row: agreement_category(
            int(row["anomaly_direction"] == "HIGH"),
            int(row["np3_high_activity"])
        ),
        axis=1
    )
)

# ------------------------------------------------------------
# 10. ML4 anomaly vs NP3 ACTIVITY_SPIKE
# ------------------------------------------------------------

ml4_np3_compare_df["ml4_vs_np3_spike"] = (
    ml4_np3_compare_df.apply(
        lambda row: agreement_category(
            int(row["anomaly_direction"] == "HIGH"),
            int(row["np3_activity_spike"])
        ),
        axis=1
    )
)

# ------------------------------------------------------------
# 11. Summary — HIGH_ACTIVITY
# ------------------------------------------------------------

high_summary = (
    ml4_np3_compare_df["ml4_vs_np3_high"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="rows")
)

high_summary["percentage"] = (
    high_summary["rows"]
    / len(ml4_np3_compare_df)
    * 100
)

high_summary = high_summary.sort_values(
    "category"
).reset_index(drop=True)

# ------------------------------------------------------------
# 12. Summary — ACTIVITY_SPIKE
# ------------------------------------------------------------

spike_summary = (
    ml4_np3_compare_df["ml4_vs_np3_spike"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="rows")
)

spike_summary["percentage"] = (
    spike_summary["rows"]
    / len(ml4_np3_compare_df)
    * 100
)

spike_summary = spike_summary.sort_values(
    "category"
).reset_index(drop=True)

# ------------------------------------------------------------
# 13. Print results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ML4 HIGH vs NP3 HIGH_ACTIVITY")
print("=" * 80)

print(
    high_summary.to_string(index=False)
)

print("\n" + "=" * 80)
print("ML4 HIGH vs NP3 ACTIVITY_SPIKE")
print("=" * 80)

print(
    spike_summary.to_string(index=False)
)

# ------------------------------------------------------------
# 14. Direction summary for ML4
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ML4 DIRECTION IN COMPARISON PERIOD")
print("=" * 80)

print(
    ml4_np3_compare_df["anomaly_direction"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# 15. Logging
# ------------------------------------------------------------

logger.info(
    "ML4 vs NP3 comparison completed successfully."
)

print("\nML4 vs NP3 comparison completed successfully.")

2026-09-05 17:26:20,458 | INFO | Starting ML4 vs NP3 comparison...
2026-09-05 17:26:20,588 | INFO | ML4 rows in comparison period: 239999



ML4 / NP3 ALIGNMENT
ML4 comparison rows: 239,999
NP3 HIGH_ACTIVITY matches: 21,440
NP3 ACTIVITY_SPIKE matches: 11,235


2026-09-05 17:26:23,308 | INFO | ML4 vs NP3 comparison completed successfully.



ML4 HIGH vs NP3 HIGH_ACTIVITY
  category   rows  percentage
BOTH_AGREE   7946    3.310847
  ML4_ONLY   7114    2.964179
   NEITHER 211445   88.102450
  NP3_ONLY  13494    5.622523

ML4 HIGH vs NP3 ACTIVITY_SPIKE
  category   rows  percentage
BOTH_AGREE   3190    1.329172
  ML4_ONLY  11870    4.945854
   NEITHER 216894   90.372877
  NP3_ONLY   8045    3.352097

ML4 DIRECTION IN COMPARISON PERIOD
anomaly_direction
HIGH       15060
LOW         2041
NORMAL    222898

ML4 vs NP3 comparison completed successfully.


In [34]:
# ============================================================
# ML4 CELL 14 — THREE-WAY ML3 / ML4 / NP3 COMPARISON
# ============================================================

logger = logging.getLogger("ML4")

logger.info("Starting three-way ML3 / ML4 / NP3 comparison...")

# ------------------------------------------------------------
# 1. Start from the validated ML3 comparison dataset
# ------------------------------------------------------------

three_way_df = ml3_compare_df[
    [
        "grid_id",
        "timestamp",
        "feature_timestamp",
        "target_activity",
        "ml_prediction_probability",
        "ml_prediction",
    ]
].copy()

three_way_df["timestamp"] = pd.to_datetime(
    three_way_df["timestamp"],
    errors="raise"
)

# ------------------------------------------------------------
# 2. Add ML4 result
# ------------------------------------------------------------

ml4_three_way = ml4_activity_df[
    [
        "grid_id",
        "timestamp",
        "total_activity",
        "historical_baseline",
        "baseline_count",
        "deviation",
        "anomaly_direction",
        "anomaly_flag",
        "reason",
    ]
].copy()

ml4_three_way["timestamp"] = pd.to_datetime(
    ml4_three_way["timestamp"],
    errors="raise"
)

three_way_df = three_way_df.merge(
    ml4_three_way,
    on=["grid_id", "timestamp"],
    how="inner",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 3. Create NP3 indicators
# ------------------------------------------------------------

np3_high_indicator = np3_alerts_df[
    np3_alerts_df["alert_type"] == "HIGH_ACTIVITY"
][
    ["grid_id", "timestamp", "baseline_activity", "reason"]
].copy()

np3_high_indicator["np3_high"] = 1

np3_high_indicator = np3_high_indicator.rename(
    columns={
        "baseline_activity": "np3_baseline_activity",
        "reason": "np3_high_reason",
    }
)

np3_spike_indicator = np3_alerts_df[
    np3_alerts_df["alert_type"] == "ACTIVITY_SPIKE"
][
    ["grid_id", "timestamp", "reason"]
].copy()

np3_spike_indicator["np3_spike"] = 1

np3_spike_indicator = np3_spike_indicator.rename(
    columns={
        "reason": "np3_spike_reason",
    }
)

# ------------------------------------------------------------
# 4. Merge NP3 HIGH_ACTIVITY
# ------------------------------------------------------------

three_way_df = three_way_df.merge(
    np3_high_indicator[
        [
            "grid_id",
            "timestamp",
            "np3_high",
            "np3_baseline_activity",
            "np3_high_reason",
        ]
    ],
    on=["grid_id", "timestamp"],
    how="left",
    validate="one_to_one"
)

three_way_df["np3_high"] = (
    three_way_df["np3_high"]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 5. Merge NP3 ACTIVITY_SPIKE
# ------------------------------------------------------------

three_way_df = three_way_df.merge(
    np3_spike_indicator[
        [
            "grid_id",
            "timestamp",
            "np3_spike",
            "np3_spike_reason",
        ]
    ],
    on=["grid_id", "timestamp"],
    how="left",
    validate="one_to_one"
)

three_way_df["np3_spike"] = (
    three_way_df["np3_spike"]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 6. Validate common comparison size
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("THREE-WAY ALIGNMENT")
print("=" * 80)

print(f"ML3 rows             : {len(ml3_compare_df):,}")
print(f"Three-way rows       : {len(three_way_df):,}")
print(f"Unique grids         : {three_way_df['grid_id'].nunique():,}")

if len(three_way_df) != len(ml3_compare_df):
    raise ValueError(
        "Three-way comparison does not contain all ML3 rows."
    )

# ------------------------------------------------------------
# 7. Create three-way comparison for NP3 HIGH_ACTIVITY
# ------------------------------------------------------------

def three_way_high(row):

    ml3 = row["ml_prediction"] == 1
    ml4 = row["anomaly_direction"] == "HIGH"
    np3 = row["np3_high"] == 1

    if ml3 and ml4 and np3:
        return "ALL_THREE"

    if ml3 and ml4 and not np3:
        return "ML3_ML4_ONLY"

    if ml3 and not ml4 and np3:
        return "ML3_NP3_ONLY"

    if not ml3 and ml4 and np3:
        return "ML4_NP3_ONLY"

    if ml3 and not ml4 and not np3:
        return "ML3_ONLY"

    if not ml3 and ml4 and not np3:
        return "ML4_ONLY"

    if not ml3 and not ml4 and np3:
        return "NP3_ONLY"

    return "NONE"


three_way_df["three_way_high_category"] = (
    three_way_df.apply(
        three_way_high,
        axis=1
    )
)

# ------------------------------------------------------------
# 8. Create three-way comparison for NP3 ACTIVITY_SPIKE
# ------------------------------------------------------------

def three_way_spike(row):

    ml3 = row["ml_prediction"] == 1
    ml4 = row["anomaly_direction"] == "HIGH"
    np3 = row["np3_spike"] == 1

    if ml3 and ml4 and np3:
        return "ALL_THREE"

    if ml3 and ml4 and not np3:
        return "ML3_ML4_ONLY"

    if ml3 and not ml4 and np3:
        return "ML3_NP3_ONLY"

    if not ml3 and ml4 and np3:
        return "ML4_NP3_ONLY"

    if ml3 and not ml4 and not np3:
        return "ML3_ONLY"

    if not ml3 and ml4 and not np3:
        return "ML4_ONLY"

    if not ml3 and not ml4 and np3:
        return "NP3_ONLY"

    return "NONE"


three_way_df["three_way_spike_category"] = (
    three_way_df.apply(
        three_way_spike,
        axis=1
    )
)

# ------------------------------------------------------------
# 9. Summaries
# ------------------------------------------------------------

high_three_way_summary = (
    three_way_df["three_way_high_category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="rows")
)

high_three_way_summary["percentage"] = (
    high_three_way_summary["rows"]
    / len(three_way_df)
    * 100
)

high_three_way_summary = high_three_way_summary.sort_values(
    "category"
).reset_index(drop=True)

spike_three_way_summary = (
    three_way_df["three_way_spike_category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="rows")
)

spike_three_way_summary["percentage"] = (
    spike_three_way_summary["rows"]
    / len(three_way_df)
    * 100
)

spike_three_way_summary = spike_three_way_summary.sort_values(
    "category"
).reset_index(drop=True)

# ------------------------------------------------------------
# 10. Print summaries
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("THREE-WAY COMPARISON — NP3 HIGH_ACTIVITY")
print("=" * 80)

print(
    high_three_way_summary.to_string(index=False)
)

print("\n" + "=" * 80)
print("THREE-WAY COMPARISON — NP3 ACTIVITY_SPIKE")
print("=" * 80)

print(
    spike_three_way_summary.to_string(index=False)
)

logger.info(
    "Three-way ML3 / ML4 / NP3 comparison completed successfully."
)

print("\nThree-way comparison completed successfully.")

2026-09-05 17:28:03,522 | INFO | Starting three-way ML3 / ML4 / NP3 comparison...



THREE-WAY ALIGNMENT
ML3 rows             : 239,976
Three-way rows       : 239,976
Unique grids         : 9,999


2026-09-05 17:28:07,035 | INFO | Three-way ML3 / ML4 / NP3 comparison completed successfully.



THREE-WAY COMPARISON — NP3 HIGH_ACTIVITY
    category   rows  percentage
   ALL_THREE   7066    2.944461
ML3_ML4_ONLY   1847    0.769660
ML3_NP3_ONLY   9361    3.900807
    ML3_ONLY  27181   11.326549
ML4_NP3_ONLY    880    0.366703
    ML4_ONLY   5264    2.193553
        NONE 184244   76.776011
    NP3_ONLY   4133    1.722256

THREE-WAY COMPARISON — NP3 ACTIVITY_SPIKE
    category   rows  percentage
   ALL_THREE   1314    0.547555
ML3_ML4_ONLY   7599    3.166567
ML3_NP3_ONLY    881    0.367120
    ML3_ONLY  35661   14.860236
ML4_NP3_ONLY   1876    0.781745
    ML4_ONLY   4268    1.778511
        NONE 181213   75.512968
    NP3_ONLY   7164    2.985299

Three-way comparison completed successfully.


In [35]:
# ============================================================
# ML4 CELL 15 — REPRESENTATIVE DISAGREEMENT ANALYSIS
# ============================================================

logger = logging.getLogger("ML4")

logger.info("Selecting representative ML3/ML4/NP3 disagreements...")

# ------------------------------------------------------------
# 1. Select useful disagreement categories
# ------------------------------------------------------------

categories_to_inspect = [
    "ALL_THREE",
    "ML3_ONLY",
    "ML4_ONLY",
    "NP3_ONLY",
    "ML3_ML4_ONLY",
    "ML4_NP3_ONLY",
]

examples = {}

for category in categories_to_inspect:

    subset = three_way_df[
        three_way_df["three_way_high_category"] == category
    ].copy()

    if len(subset) == 0:
        continue

    # Select examples with strongest available signal
    if category == "ML3_ONLY":
        subset = subset.sort_values(
            "ml_prediction_probability",
            ascending=False
        )

    elif category == "ML4_ONLY":
        subset = subset.sort_values(
            "deviation",
            ascending=False
        )

    elif category == "NP3_ONLY":
        subset = subset.sort_values(
            "np3_baseline_activity",
            ascending=False
        )

    else:
        subset = subset.sort_values(
            "ml_prediction_probability",
            ascending=False
        )

    examples[category] = subset.head(5)

# ------------------------------------------------------------
# 2. Display representative observations
# ------------------------------------------------------------

display_columns = [
    "grid_id",
    "timestamp",
    "target_activity",
    "ml_prediction_probability",
    "ml_prediction",
    "total_activity",
    "historical_baseline",
    "deviation",
    "anomaly_direction",
    "np3_high",
    "np3_baseline_activity",
    "three_way_high_category",
]

for category, subset in examples.items():

    available_columns = [
        col for col in display_columns
        if col in subset.columns
    ]

    print("\n" + "=" * 100)
    print(f"REPRESENTATIVE CATEGORY: {category}")
    print("=" * 100)

    print(
        subset[available_columns]
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 3. Automatically generate interpretation guidance
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("INTERPRETATION")
print("=" * 100)

print("""
ML3_ONLY:
ML3 predicts high activity for the next hour, while ML4 does not
classify the target hour as HIGH relative to its historical
grid/hour-of-day baseline. This is consistent with ML3 being a
predictive model and ML4 being a current/historical anomaly detector.

ML4_ONLY:
ML4 identifies an unusually high current observation relative to
historical grid/hour behavior, while ML3 does not predict the
observation as a high-activity event. This indicates a current
historical deviation that did not translate into the ML3 classifier's
positive prediction.

NP3_ONLY:
NP3 records a rule-based HIGH_ACTIVITY alert while ML4 and/or ML3
do not. NP3 uses its own within-day rule and activity-floor logic,
so this is not necessarily an error in either method.

ML3_ML4_ONLY:
Both ML3 and ML4 identify a high-activity situation, while NP3 does
not record HIGH_ACTIVITY. This demonstrates agreement between the
predictive and historical-anomaly approaches despite different
rule definitions.

ML4_NP3_ONLY:
ML4 and NP3 identify high activity while ML3 does not predict the
event. This represents a current operational signal that was not
classified as a next-hour high-activity risk by ML3.

ALL_THREE:
All three approaches identify the same high-activity observation,
providing the strongest agreement between predictive, anomaly-based,
and rule-based detection.
""")

logger.info(
    "Representative disagreement analysis completed."
)

print("\nRepresentative disagreement analysis completed successfully.")

2026-09-05 17:29:58,173 | INFO | Selecting representative ML3/ML4/NP3 disagreements...
2026-09-05 17:29:58,271 | INFO | Representative disagreement analysis completed.



REPRESENTATIVE CATEGORY: ALL_THREE
 grid_id           timestamp  target_activity  ml_prediction_probability  ml_prediction  total_activity  historical_baseline  deviation anomaly_direction  np3_high  np3_baseline_activity three_way_high_category
    7360 2013-11-07 16:00:00        1682.0126                   0.952483              1       1682.0126            803.47610   1.093420              HIGH         1              1119.9921               ALL_THREE
    7149 2013-11-07 15:00:00         396.3903                   0.952483              1        396.3903            198.04030   1.001564              HIGH         1               213.0957               ALL_THREE
    7250 2013-11-07 08:00:00         837.2956                   0.952483              1        837.2956            477.78015   0.752470              HIGH         1               425.5894               ALL_THREE
    7725 2013-11-07 08:00:00         384.2555                   0.952483              1        384.2555            194.0

In [36]:
# ============================================================
# ML4 CELL 16 — SAVE FINAL ANOMALY DATASET
# ============================================================

import os

ML4_DATASET_PATH = "data/ml4_anomaly_dataset.csv"

os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

logger.info("Saving final ML4 anomaly dataset...")

# ------------------------------------------------------------
# Final columns
# ------------------------------------------------------------

ML4_FINAL_COLUMNS = [
    "grid_id",
    "timestamp",
    "hour_of_day",
    "total_activity",
    "historical_baseline",
    "baseline_count",
    "deviation",
    "anomaly_score",
    "anomaly_direction",
    "anomaly_flag",
    "reason",
    "recommended_action",
]

missing_final_columns = [
    col
    for col in ML4_FINAL_COLUMNS
    if col not in ml4_activity_df.columns
]

if missing_final_columns:
    raise ValueError(
        f"Missing final ML4 columns: {missing_final_columns}"
    )

ml4_final_df = ml4_activity_df[
    ML4_FINAL_COLUMNS
].copy()

# ------------------------------------------------------------
# Ensure timestamp is explicitly datetime
# ------------------------------------------------------------

ml4_final_df["timestamp"] = pd.to_datetime(
    ml4_final_df["timestamp"],
    errors="raise"
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

ml4_final_df.to_csv(
    ML4_DATASET_PATH,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

file_size_mb = (
    os.path.getsize(ML4_DATASET_PATH)
    / (1024 ** 2)
)

logger.info(
    "Final ML4 dataset saved successfully: %.2f MB",
    file_size_mb
)

print("\n" + "=" * 80)
print("ML4 FINAL DATASET SAVED")
print("=" * 80)

print(f"Path       : {ML4_DATASET_PATH}")
print(f"Rows       : {len(ml4_final_df):,}")
print(f"Columns    : {len(ml4_final_df.columns)}")
print(f"Size       : {file_size_mb:.2f} MB")
print(
    f"Timestamp  : "
    f"{ml4_final_df['timestamp'].min()} "
    f"to "
    f"{ml4_final_df['timestamp'].max()}"
)

print("\nAnomaly distribution:")
print(
    ml4_final_df["anomaly_direction"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nAction distribution:")
print(
    ml4_final_df["recommended_action"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nML4 final dataset saved successfully.")

2026-09-05 17:30:18,661 | INFO | Saving final ML4 anomaly dataset...
2026-09-05 17:30:28,550 | INFO | Final ML4 dataset saved successfully: 322.71 MB



ML4 FINAL DATASET SAVED
Path       : data/ml4_anomaly_dataset.csv
Rows       : 1,679,994
Columns    : 12
Size       : 322.71 MB
Timestamp  : 2013-11-01 00:00:00 to 2013-11-07 23:00:00

Anomaly distribution:
anomaly_direction
HIGH       174321
LOW          7888
NORMAL     777785
UNKNOWN    720000

Action distribution:
recommended_action
INVESTIGATE     182209
NONE           1497785

ML4 final dataset saved successfully.


In [37]:
# ============================================================
# ML4 CELL 17 — SAVE FINAL ML4 METADATA
# ============================================================

import os
import json
from datetime import datetime

logger = logging.getLogger("ML4")

ML4_METADATA_PATH = "models/ml4_anomaly_metadata.json"

os.makedirs("models", exist_ok=True)

logger.info("Creating final ML4 metadata...")

# ------------------------------------------------------------
# 1. Calculate final ML4 statistics
# ------------------------------------------------------------

direction_counts = (
    ml4_final_df["anomaly_direction"]
    .value_counts()
    .to_dict()
)

flag_counts = (
    ml4_final_df["anomaly_flag"]
    .value_counts()
    .to_dict()
)

action_counts = (
    ml4_final_df["recommended_action"]
    .value_counts()
    .to_dict()
)

# ------------------------------------------------------------
# 2. Three-way comparison summaries
# ------------------------------------------------------------

three_way_high_records = {}

if "high_three_way_summary" in globals():

    for _, row in high_three_way_summary.iterrows():
        three_way_high_records[str(row["category"])] = {
            "rows": int(row["rows"]),
            "percentage": float(row["percentage"])
        }

three_way_spike_records = {}

if "spike_three_way_summary" in globals():

    for _, row in spike_three_way_summary.iterrows():
        three_way_spike_records[str(row["category"])] = {
            "rows": int(row["rows"]),
            "percentage": float(row["percentage"])
        }

# ------------------------------------------------------------
# 3. ML3 vs ML4 comparison summary
# ------------------------------------------------------------

ml3_ml4_summary_records = {}

if "high_group_summary" in globals():

    for _, row in high_group_summary.iterrows():
        ml3_ml4_summary_records[str(row["group"])] = {
            "rows": int(row["rows"]),
            "percentage": float(row["percentage"])
        }

# ------------------------------------------------------------
# 4. ML4 vs NP3 summaries
# ------------------------------------------------------------

ml4_np3_high_records = {}

if "high_summary" in globals():

    for _, row in high_summary.iterrows():
        ml4_np3_high_records[str(row["category"])] = {
            "rows": int(row["rows"]),
            "percentage": float(row["percentage"])
        }

ml4_np3_spike_records = {}

if "spike_summary" in globals():

    for _, row in spike_summary.iterrows():
        ml4_np3_spike_records[str(row["category"])] = {
            "rows": int(row["rows"]),
            "percentage": float(row["percentage"])
        }

# ------------------------------------------------------------
# 5. Final metadata
# ------------------------------------------------------------

metadata = {
    "project_stage": "ML4",
    "task": "Anomaly Detection",
    "created_at": datetime.now().isoformat(),

    "data": {
        "source": "fact_network_activity",
        "activity_column": "total_activity",
        "input_rows": int(len(ml4_final_df)),
        "grid_count": int(
            ml4_final_df["grid_id"].nunique()
        ),
        "timestamp_min": str(
            ml4_final_df["timestamp"].min()
        ),
        "timestamp_max": str(
            ml4_final_df["timestamp"].max()
        )
    },

    "baseline": {
        "statistic": "median",
        "keys": [
            "grid_id",
            "hour_of_day"
        ],
        "history_definition": (
            "Observations strictly before the current timestamp "
            "within the same grid_id and hour_of_day bucket."
        ),
        "minimum_history_count": 3,
        "current_value": "total_activity"
    },

    "anomaly_calculation": {
        "deviation": (
            "(current_activity - historical_baseline) "
            "/ historical_baseline"
        ),
        "anomaly_score": "abs(deviation)",
        "threshold": 0.50,
        "threshold_description": "±50%"
    },

    "classification": {
        "HIGH": (
            "deviation >= +50%"
        ),
        "LOW": (
            "deviation <= -50%"
        ),
        "NORMAL": (
            "deviation within ±50%"
        ),
        "UNKNOWN": (
            "Insufficient historical observations "
            "to calculate a baseline."
        )
    },

    "operational_action": {
        "anomaly": "INVESTIGATE",
        "normal": "NONE",
        "interpretation": (
            "An anomaly is an operational attention signal, "
            "not confirmation of network congestion."
        )
    },

    "feature_engineering": {
        "baseline_features": [
            "grid_id",
            "hour_of_day",
            "historical_baseline",
            "baseline_count"
        ],
        "derived_features": [
            "deviation",
            "anomaly_score",
            "anomaly_direction",
            "anomaly_flag"
        ]
    },

    "validation": {
        "null_total_activity": int(
            ml4_final_df["total_activity"].isna().sum()
        ),
        "null_historical_baseline": int(
            ml4_final_df["historical_baseline"].isna().sum()
        ),
        "null_deviation": int(
            ml4_final_df["deviation"].isna().sum()
        ),
        "null_reason": int(
            ml4_final_df["reason"].isna().sum()
        ),
        "null_recommended_action": int(
            ml4_final_df["recommended_action"].isna().sum()
        ),
        "duplicate_grid_timestamp": int(
            ml4_final_df.duplicated(
                subset=["grid_id", "timestamp"]
            ).sum()
        )
    },

    "distribution": {
        "anomaly_direction": {
            str(k): int(v)
            for k, v in direction_counts.items()
        },
        "anomaly_flag": {
            str(k): int(v)
            for k, v in flag_counts.items()
        },
        "recommended_action": {
            str(k): int(v)
            for k, v in action_counts.items()
        }
    },

    "comparison": {
        "ml3_vs_ml4_high_activity": ml3_ml4_summary_records,

        "ml4_vs_np3_high_activity": ml4_np3_high_records,

        "ml4_vs_np3_activity_spike": ml4_np3_spike_records,

        "three_way_ml3_ml4_np3_high_activity":
            three_way_high_records,

        "three_way_ml3_ml4_np3_activity_spike":
            three_way_spike_records
    },

    "interpretation_notes": [
        "ML4 is a descriptive/current anomaly detector.",
        "ML3 is a predictive next-hour high-activity classifier.",
        "NP3 is rule-based and uses separate alert definitions.",
        "NP3 HIGH_ACTIVITY and ACTIVITY_SPIKE represent different signals.",
        "Agreement between methods is informative but disagreement is expected.",
        "Absence of an NP3 alert does not necessarily mean the mathematical "
        "condition was false because NP3 applies its configured activity floor.",
        "High activity must not be interpreted as confirmed network congestion "
        "because the dataset does not contain capacity, throughput, latency, "
        "or similar network-performance measurements."
    ],

    "artifacts": {
        "dataset": "data/ml4_anomaly_dataset.csv",
        "metadata": "models/ml4_anomaly_metadata.json",
        "ml3_dataset": "data/ml3_final_engineered_dataset.csv",
        "ml3_model": "models/ml3_high_activity_decision_tree.joblib",
        "ml3_metadata": "models/ml3_model_metadata.json"
    }
}

# ------------------------------------------------------------
# 6. Save metadata
# ------------------------------------------------------------

with open(
    ML4_METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 7. Verify saved file
# ------------------------------------------------------------

if not os.path.exists(ML4_METADATA_PATH):
    raise FileNotFoundError(
        "ML4 metadata file was not created."
    )

file_size_kb = (
    os.path.getsize(ML4_METADATA_PATH)
    / 1024
)

logger.info(
    "ML4 metadata saved successfully: %.2f KB",
    file_size_kb
)

print("\n" + "=" * 80)
print("ML4 METADATA SAVED")
print("=" * 80)

print(f"Path : {ML4_METADATA_PATH}")
print(f"Size : {file_size_kb:.2f} KB")

print("\nML4 metadata successfully saved.")

2026-09-05 17:31:02,349 | INFO | Creating final ML4 metadata...
2026-09-05 17:31:02,578 | INFO | ML4 metadata saved successfully: 7.15 KB



ML4 METADATA SAVED
Path : models/ml4_anomaly_metadata.json
Size : 7.15 KB

ML4 metadata successfully saved.
